In [1]:
# ============================================================
# STEP 1: Install Demucs and verify it works
# ============================================================

import subprocess
import sys

print("Installing Demucs...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "demucs", "--quiet"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✓ Demucs installed")
else:
    print(f"✗ Failed: {result.stderr[-200:]}")

# Verify
try:
    import demucs
    print(f"✓ Demucs version: {demucs.__version__}")
except Exception as e:
    print(f"✗ Import failed: {e}")

Installing Demucs...
✓ Demucs installed
✓ Demucs version: 4.0.1


In [2]:
# ============================================================
# STEP 1B: Test Demucs on one song
# ============================================================

import torch
import torchaudio
from pathlib import Path

PROJECT = Path("C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION")
DATASET = Path("D:/dataset")

# Load one test song
test_song_path = next((DATASET / "test").iterdir())
mixture_path   = test_song_path / "mixture.wav"

print(f"Test song: {test_song_path.name}")
print(f"Mixture  : {mixture_path}")

# Load audio
audio, sr = torchaudio.load(str(mixture_path), num_frames=44100 * 10)
print(f"Audio shape: {audio.shape}  SR: {sr}")

# Run Demucs
print("\nRunning Demucs htdemucs (4-stem model)...")

from demucs.pretrained import get_model
from demucs.apply import apply_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained htdemucs (best Demucs model)
demucs_model = get_model("htdemucs")
demucs_model.to(device)
demucs_model.eval()

print(f"✓ Loaded htdemucs")
print(f"  Sources: {demucs_model.sources}")  # ['drums', 'bass', 'other', 'vocals']

# Separate
audio_batch = audio.unsqueeze(0).to(device)   # [1, 2, T]

with torch.no_grad():
    sources = apply_model(
        demucs_model, audio_batch,
        progress=True, num_workers=0,
    )
# sources shape: [1, 4, 2, T]  (batch, stems, channels, time)

print(f"\n✓ Separation complete!")
print(f"  Output shape: {sources.shape}")
print(f"  Stems: {demucs_model.sources}")

# Save each stem
out_dir = PROJECT / "outputs" / "audio" / "demucs_test"
out_dir.mkdir(parents=True, exist_ok=True)

stem_names = demucs_model.sources
for i, stem in enumerate(stem_names):
    stem_audio = sources[0, i].cpu()   # [2, T]
    out_path   = out_dir / f"{stem}.wav"
    torchaudio.save(str(out_path), stem_audio, sr)
    print(f"  Saved: {out_path.name}")

print(f"\n✓ Demucs test complete!")
print(f"  Listen to files in: {out_dir}")


Test song: Al James - Schoolboy Facination
Mixture  : D:\dataset\test\Al James - Schoolboy Facination\mixture.wav
Audio shape: torch.Size([2, 441000])  SR: 44100

Running Demucs htdemucs (4-stem model)...


Downloading: "https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/955717e8-8726e21a.th" to C:\Users\Disha Saini/.cache\torch\hub\checkpoints\955717e8-8726e21a.th
100%|█████████████████████████████████████████████████████████████████████████████| 80.2M/80.2M [00:22<00:00, 3.76MB/s]


✓ Loaded htdemucs
  Sources: ['drums', 'bass', 'other', 'vocals']


100%|██████████████████████████████████████████████████████████████████████████| 11.7/11.7 [00:02<00:00,  5.69seconds/s]


✓ Separation complete!
  Output shape: torch.Size([1, 4, 2, 441000])
  Stems: ['drums', 'bass', 'other', 'vocals']
  Saved: drums.wav
  Saved: bass.wav
  Saved: other.wav
  Saved: vocals.wav

✓ Demucs test complete!
  Listen to files in: C:\Users\Disha Saini\Documents\ML\AUDIO_SEPARATION\outputs\audio\demucs_test


In [3]:
# ============================================================
# PHASE 5: IMPROVED GSN VOCALS WITH KNOWLEDGE DISTILLATION
# ============================================================
# What this does differently from all previous training:
#
#   1. LOG-MAGNITUDE INPUT
#      Old: model sees raw magnitude (values 0 to ~100)
#      New: model sees log1p(magnitude) (values 0 to ~5)
#      Why: log-magnitude matches human hearing, reduces
#           dominance of loud frequencies, less muffling
#
#   2. KNOWLEDGE DISTILLATION FROM DEMUCS
#      Old: learn from noisy ground truth only
#      New: also learn from clean Demucs predictions
#      Why: Demucs output has less bleed and better phase
#           than raw ground truth stems. Teaching our model
#           to match Demucs output teaches it cleanliness.
#
#   3. ENERGY CONSTRAINT
#      Old: model could output near-silence freely
#      New: predicted energy must match target energy
#      Why: prevents the silence collapse we kept seeing
#
#   4. MULTI-RESOLUTION STFT LOSS
#      Old: L1 on time-domain audio only
#      New: L1 on log-magnitude at 1024 and 2048 FFT sizes
#      Why: captures both fine and coarse spectral structure,
#           directly reduces muffling
#
#   5. VOCALS ONLY (no multi-stem)
#      Demucs handles drums/bass/other reliably.
#      We focus 100% of training on making vocals excellent.
# ============================================================

import os
import sys
import gc
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader

# ── Paths ────────────────────────────────────────────────────
PROJECT = Path("C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION")
DATASET = Path("D:/dataset")

sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

for k in list(sys.modules.keys()):
    if "src" in k:
        del sys.modules[k]

gc.collect()
torch.cuda.empty_cache()
torch.manual_seed(42)

# ── Imports ───────────────────────────────────────────────────
from src.models.unet import UNetConfig, EncoderBlock, DecoderBlock
from src.models.harmonic_graph import build_harmonic_edges
from src.models.gcn_bottleneck import HarmonicGCNBottleneck
from src.data.musdb_dataset import DataConfig, MUSDB18Dataset
from src.audio.dsp import AudioAugmenter
from src.training.losses import compute_si_sdr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── Install Demucs if needed ──────────────────────────────────
try:
    import demucs
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "demucs", "-q"])

from demucs.pretrained import get_model
from demucs.apply import apply_model

# ============================================================
# STFT
# ============================================================
class AudioSTFT(nn.Module):
    def __init__(self, n_fft=2048, hop_length=512):
        super().__init__()
        self.n_fft      = n_fft
        self.hop_length = hop_length
        self.register_buffer("window", torch.hann_window(n_fft))

    def encode(self, audio):
        """[B, 2, T] -> magnitude [B, 2, F, T], phase [B, 2, F, T], length"""
        B, C, T = audio.shape
        flat    = audio.reshape(B * C, T)
        spec    = torch.stft(
            flat, self.n_fft, self.hop_length,
            window=self.window, center=True, return_complex=True,
        )
        _, F, Tf = spec.shape
        spec = spec.reshape(B, C, F, Tf)
        return spec.abs(), torch.angle(spec), T

    def decode(self, magnitude, phase, length=None):
        """magnitude [B, 2, F, T], phase [B, 2, F, T] -> [B, 2, T]"""
        B, C, F, Tf = magnitude.shape
        cmplx = magnitude * torch.exp(1j * phase)
        flat  = cmplx.reshape(B * C, F, Tf)
        audio = torch.istft(
            flat, self.n_fft, self.hop_length,
            window=self.window, center=True, length=length,
        )
        return audio.reshape(B, C, -1)


# ============================================================
# KNOWLEDGE DISTILLATION LOSS
# ============================================================
class DistillationLoss(nn.Module):
    """
    Four-component loss designed to fix muffled output.

    Component breakdown:
      w_gt=0.4     L1 vs ground truth vocals
                   The "correct answer" signal

      w_kd=0.3     L1 vs Demucs prediction
                   The "clean sound" teacher signal
                   Demucs output is less muffled than GT

      w_energy=0.1 |mean(|pred|) - mean(|target|)|
                   Prevents the model from outputting silence

      w_stft=0.2   Log-magnitude L1 at 1024 and 2048 FFT
                   Directly penalises muffled spectrograms
    """

    def __init__(
        self,
        w_gt     : float = 0.4,
        w_kd     : float = 0.3,
        w_energy : float = 0.1,
        w_stft   : float = 0.2,
    ):
        super().__init__()
        self.w_gt     = w_gt
        self.w_kd     = w_kd
        self.w_energy = w_energy
        self.w_stft   = w_stft

        self.register_buffer("win_1024", torch.hann_window(1024))
        self.register_buffer("win_2048", torch.hann_window(2048))

    def _log_stft_l1(self, pred, target, n_fft, hop, win):
        p   = pred.reshape(-1, pred.shape[-1])
        t   = target.reshape(-1, target.shape[-1])
        p_m = torch.stft(
            p, n_fft, hop, n_fft, win,
            return_complex=True, center=True,
        ).abs()
        t_m = torch.stft(
            t, n_fft, hop, n_fft, win,
            return_complex=True, center=True,
        ).abs()
        return F.l1_loss(torch.log1p(p_m), torch.log1p(t_m))

    def forward(
        self,
        pred_audio    : torch.Tensor,
        target_audio  : torch.Tensor,
        demucs_audio  : torch.Tensor,
    ) -> tuple:
        l_gt     = F.l1_loss(pred_audio, target_audio)
        l_kd     = F.l1_loss(pred_audio, demucs_audio.detach())
        l_energy = (
            pred_audio.abs().mean() - target_audio.abs().mean()
        ).abs()
        l_stft   = (
            self._log_stft_l1(pred_audio, target_audio, 1024, 256, self.win_1024) +
            self._log_stft_l1(pred_audio, target_audio, 2048, 512, self.win_2048)
        ) / 2.0

        total = (
            self.w_gt     * l_gt     +
            self.w_kd     * l_kd     +
            self.w_energy * l_energy +
            self.w_stft   * l_stft
        )

        return total, {
            "total" : total.item(),
            "gt"    : l_gt.item(),
            "kd"    : l_kd.item(),
            "energy": l_energy.item(),
            "stft"  : l_stft.item(),
        }


# ============================================================
# GSN MODEL — log-magnitude input, magnitude mask output
# ============================================================
class ImprovedGSN(nn.Module):
    """
    Improved GSN for Phase 5.

    Changes from Phase 2/3:
      - Input: log1p(magnitude) instead of magnitude
      - Output: magnitude mask [0, 1] — same as before
      - Architecture: identical H-GCN U-Net
      - Training: distillation from Demucs teacher

    The log-magnitude input is the key fix for muffling.
    When the model sees raw magnitude, large values dominate.
    When it sees log-magnitude, all frequency bins contribute
    equally to the loss, so the model learns fine details.
    """

    def __init__(
        self,
        n_freq_bins   : int   = 1025,
        base_channels : int   = 32,
        depth         : int   = 4,
        dropout       : float = 0.1,
    ):
        super().__init__()
        ch = [base_channels * (2 ** i) for i in range(depth)]

        # Encoder — 1 channel input (log-magnitude)
        self.encoders = nn.ModuleList()
        self.encoders.append(EncoderBlock(1, ch[0], dropout, True))
        for i in range(1, depth):
            self.encoders.append(
                EncoderBlock(ch[i-1], ch[i], dropout, True)
            )

        # Harmonic GCN bottleneck
        bn_freq  = max(n_freq_bins // (2 ** depth), 4)
        bn_nfft  = max(2048 // (2 ** depth), 64)
        edge_idx = build_harmonic_edges(bn_freq, 44100, bn_nfft, 5)
        self.bottleneck = HarmonicGCNBottleneck(ch[-1], edge_idx, dropout)

        # Decoder
        self.decoders = nn.ModuleList()
        for i in range(depth - 1, 0, -1):
            self.decoders.append(
                DecoderBlock(ch[i], ch[i], ch[i-1], dropout)
            )
        self.decoders.append(
            DecoderBlock(ch[0], ch[0], ch[0], dropout)
        )

        # Magnitude mask — sigmoid output in (0, 1)
        self.mask_head = nn.Sequential(
            nn.Conv2d(ch[0], 1, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, log_magnitude: torch.Tensor) -> torch.Tensor:
        """
        log_magnitude : [B, 1, F, T]  — log1p(magnitude)
        returns mask  : [B, 1, F, T]  — in (0, 1)
        """
        x     = log_magnitude
        skips = []

        for enc in self.encoders:
            x, skip = enc(x)
            skips.append(skip)

        x = self.bottleneck(x)

        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)

        return self.mask_head(x)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def separate(
        self,
        mixture_audio  : torch.Tensor,
        stft_processor : "AudioSTFT",
    ) -> torch.Tensor:
        """
        Full separation pipeline.

        mixture_audio : [B, 2, T]
        returns       : [B, 2, T]
        """
        mag, phase, T = stft_processor.encode(mixture_audio)
        log_mag       = torch.log1p(mag)

        mask_l = self(log_mag[:, 0:1])
        mask_r = self(log_mag[:, 1:2])
        mask   = torch.cat([mask_l, mask_r], dim=1)

        pred_mag = mag * mask
        return stft_processor.decode(pred_mag, phase, length=T)


# ============================================================
# DEMUCS TEACHER
# ============================================================
print("\nLoading Demucs teacher (htdemucs)...")
demucs_teacher = get_model("htdemucs")
demucs_teacher.to(device)
demucs_teacher.eval()
for p in demucs_teacher.parameters():
    p.requires_grad = False

DEMUCS_SOURCES = demucs_teacher.sources
VOCALS_IDX     = DEMUCS_SOURCES.index("vocals")
print(f"✓ Demucs loaded | stems={DEMUCS_SOURCES} | vocals_idx={VOCALS_IDX}")


def get_demucs_vocals(mixture_audio: torch.Tensor) -> torch.Tensor:
    """
    Get Demucs vocal prediction.
    mixture_audio : [B, 2, T]
    returns       : [B, 2, T]
    """
    with torch.no_grad():
        sources = apply_model(
            demucs_teacher, mixture_audio,
            progress=False, num_workers=0,
        )
    return sources[:, VOCALS_IDX, :, :]


# ============================================================
# BUILD EVERYTHING
# ============================================================
stft    = AudioSTFT().to(device)
model   = ImprovedGSN().to(device)
loss_fn = DistillationLoss().to(device)

print(f"\nGSN parameters: {model.count_parameters():,}")

# Load Phase 2 weights as starting point
# Phase 2 is a clean unbiased separator — good foundation
p2_path = PROJECT / "checkpoints" / "phase2" / "phase2_final_3.12dB.pt"
if p2_path.exists():
    p2_state = torch.load(p2_path, map_location=device)["model_state"]
    m_state  = model.state_dict()
    loaded   = 0
    for k, v in p2_state.items():
        if k in m_state and m_state[k].shape == v.shape:
            m_state[k] = v
            loaded += 1
    model.load_state_dict(m_state)
    print(f"Loaded {loaded} Phase 2 tensors")
else:
    print("No Phase 2 checkpoint — training from scratch")

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(), lr=3e-4, weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode     = "max",    # maximize SI-SDR
    patience = 4,
    factor   = 0.5,
    min_lr   = 1e-6,
)

# Data
data_cfg = DataConfig(
    dataset_path   = str(DATASET),
    sample_rate    = 44100,
    chunk_duration = 4.0,
    batch_size     = 2,
    num_workers    = 0,
)
aug = AudioAugmenter(gain_range=(0.7, 1.3), swap_prob=0.5, seed=42)

train_ds     = MUSDB18Dataset(data_cfg, "train", "vocals", aug)
val_ds       = MUSDB18Dataset(data_cfg, "test",  "vocals", None)

train_loader = DataLoader(
    train_ds, batch_size=2, shuffle=True,  num_workers=0, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=2, shuffle=False, num_workers=0,
)

# Checkpointing
save_dir  = PROJECT / "checkpoints" / "phase5"
save_dir.mkdir(parents=True, exist_ok=True)
best_path = save_dir / "best_vocal_model.pt"

def save_ckpt(epoch: int, val_sdr: float, history: dict) -> None:
    torch.save({
        "epoch"      : epoch,
        "model_state": model.state_dict(),
        "val_si_sdr" : val_sdr,
        "history"    : history,
    }, best_path)

def cpu_sdr(pred: torch.Tensor, target: torch.Tensor) -> float:
    return compute_si_sdr(pred.detach().cpu(), target.detach().cpu())

# ============================================================
# SANITY CHECK
# ============================================================
print("\nSanity check...")
with torch.no_grad():
    m, t = next(iter(train_loader))
    m, t = m.to(device), t.to(device)
    mag, phase, T = stft.encode(m)
    log_mag       = torch.log1p(mag)
    mask_l        = model(log_mag[:, 0:1])
    mask_r        = model(log_mag[:, 1:2])
    mask          = torch.cat([mask_l, mask_r], dim=1)
    pred_mag      = mag * mask
    pred_aud      = stft.decode(pred_mag, phase, length=T)
    demucs_vocals = get_demucs_vocals(m)
    loss, comps   = loss_fn(pred_aud, t, demucs_vocals)
    sdr           = cpu_sdr(pred_aud, t)

print(f"  Mask range : [{mask.min():.3f}, {mask.max():.3f}]")
print(f"  Loss       : {loss.item():.4f}")
print(f"  SDR        : {sdr:+.2f} dB")
print(f"  Components : " + "  ".join(f"{k}={v:.3f}" for k, v in comps.items()))
assert loss.item() < 10.0, f"Sanity check failed: loss={loss.item()}"
print("  ✓ Passed\n")

del m, t, mag, phase, log_mag, mask_l, mask_r, mask, pred_mag, pred_aud, demucs_vocals
gc.collect()
torch.cuda.empty_cache()

# ============================================================
# TRAINING
# ============================================================
MAX_EPOCHS  = 20
LOG_EVERY   = 25
VAL_BATCHES = 15

print("=" * 60)
print("PHASE 5 — IMPROVED GSN WITH KNOWLEDGE DISTILLATION")
print(f"  Teacher  : Demucs htdemucs (frozen)")
print(f"  Input    : log-magnitude spectrogram")
print(f"  Loss     : GT + KD + Energy + log-STFT")
print(f"  Epochs   : {MAX_EPOCHS}")
print(f"  Batches  : {len(train_loader)} train | {VAL_BATCHES} val")
print("=" * 60 + "\n")

best_val_sdr = float("-inf")
no_improve   = 0
history      = {"train_loss": [], "val_sdr": [], "lr": []}

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    demucs_teacher.eval()

    t_loss_sum  = 0.0
    t_sdr_sum   = 0.0
    t_sdr_count = 0
    t0          = time.time()

    for step, (mix_aud, tgt_aud) in enumerate(train_loader):
        mix_aud = mix_aud.to(device)   # [B, 2, T]
        tgt_aud = tgt_aud.to(device)

        # Get Demucs teacher prediction (no grad)
        demucs_vocals = get_demucs_vocals(mix_aud)

        # STFT + log-magnitude
        mag, phase, T = stft.encode(mix_aud)
        log_mag       = torch.log1p(mag)

        # Forward pass
        mask_l   = model(log_mag[:, 0:1])
        mask_r   = model(log_mag[:, 1:2])
        mask     = torch.cat([mask_l, mask_r], dim=1)
        pred_mag = mag * mask
        pred_aud = stft.decode(pred_mag, phase, length=T)

        # Loss
        loss, comps = loss_fn(pred_aud, tgt_aud, demucs_vocals)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        t_loss_sum += loss.item()

        if step % LOG_EVERY == 0:
            sdr = cpu_sdr(pred_aud, tgt_aud)
            t_sdr_sum   += sdr
            t_sdr_count += 1
            eta = (time.time() - t0) / (step + 1) * (len(train_loader) - step - 1)
            lr  = optimizer.param_groups[0]["lr"]
            print(
                f"  E{epoch:02d} [{step:03d}/{len(train_loader)}] "
                f"loss={loss.item():.4f} "
                f"(gt={comps['gt']:.3f} "
                f"kd={comps['kd']:.3f} "
                f"energy={comps['energy']:.3f} "
                f"stft={comps['stft']:.3f}) "
                f"sdr={sdr:+.2f}dB "
                f"lr={lr:.1e} "
                f"ETA={eta:.0f}s"
            )

        del mix_aud, tgt_aud, demucs_vocals
        del mag, phase, log_mag, mask_l, mask_r, mask
        del pred_mag, pred_aud, loss

    avg_t_loss = t_loss_sum / len(train_loader)
    avg_t_sdr  = t_sdr_sum  / max(1, t_sdr_count)

    # ── Validate ──────────────────────────────────────────────
    model.eval()
    v_sdr_sum = 0.0
    v_count   = 0

    with torch.no_grad():
        for vi, (mix_aud, tgt_aud) in enumerate(val_loader):
            if vi >= VAL_BATCHES:
                break
            mix_aud = mix_aud.to(device)
            tgt_aud = tgt_aud.to(device)

            mag, phase, T = stft.encode(mix_aud)
            log_mag       = torch.log1p(mag)

            mask_l   = model(log_mag[:, 0:1])
            mask_r   = model(log_mag[:, 1:2])
            mask     = torch.cat([mask_l, mask_r], dim=1)
            pred_mag = mag * mask
            pred_aud = stft.decode(pred_mag, phase, length=T)

            sdr        = cpu_sdr(pred_aud, tgt_aud)
            v_sdr_sum += sdr
            v_count   += 1

            del mix_aud, tgt_aud, mag, phase, log_mag
            del mask_l, mask_r, mask, pred_mag, pred_aud

    avg_val_sdr = v_sdr_sum / max(1, v_count)
    lr          = optimizer.param_groups[0]["lr"]
    scheduler.step(avg_val_sdr)

    history["train_loss"].append(avg_t_loss)
    history["val_sdr"].append(avg_val_sdr)
    history["lr"].append(lr)

    elapsed = time.time() - t0

    print(f"\n{'='*60}")
    print(f"EPOCH {epoch}/{MAX_EPOCHS}  ({elapsed:.0f}s)  lr={lr:.1e}")
    print(f"  Train : loss={avg_t_loss:.4f}  sdr={avg_t_sdr:+.2f} dB")
    print(f"  Val   : sdr={avg_val_sdr:+.2f} dB")
    print(f"{'='*60}")

    if avg_val_sdr > best_val_sdr:
        best_val_sdr = avg_val_sdr
        no_improve   = 0
        save_ckpt(epoch, avg_val_sdr, history)
        print(f"★ New best: {best_val_sdr:+.2f} dB  (saved)\n")
    else:
        no_improve += 1
        print(f"  No improvement ({no_improve}/8)\n")
        if no_improve >= 8:
            print("Early stopping.")
            break

print("=" * 60)
print("PHASE 5 COMPLETE")
print("=" * 60)
print(f"  Phase 2 baseline : +3.12 dB")
print(f"  Phase 3 CLAP     : +3.80 dB")
print(f"  Phase 5 result   : {best_val_sdr:+.2f} dB")
print()
if best_val_sdr > 3.80:
    print("✓ Improvement over Phase 3!")
    print("  → Proceed to Phase 6: Hybrid System")
else:
    print("Run again to resume training from checkpoint.")

Device : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM   : 6.4 GB

Loading Demucs teacher (htdemucs)...
✓ Demucs loaded | stems=['drums', 'bass', 'other', 'vocals'] | vocals_idx=3

GSN parameters: 2,301,634
Loaded 106 Phase 2 tensors

MUSDB18Dataset  split=train  target=vocals
  train: scanning 100 folders …
  train: 100 valid songs
  100 songs × 10 = 1000 items  | chunk=4.0s (176400 samples)

MUSDB18Dataset  split=test  target=vocals
  test: scanning 50 folders …
  test: 50 valid songs
  50 songs × 1 = 50 items  | chunk=4.0s (176400 samples)

Sanity check...
  Mask range : [0.394, 0.976]
  Loss       : 0.0718
  SDR        : -9.40 dB
  Components : total=0.072  gt=0.064  kd=0.059  energy=0.017  stft=0.135
  ✓ Passed

PHASE 5 — IMPROVED GSN WITH KNOWLEDGE DISTILLATION
  Teacher  : Demucs htdemucs (frozen)
  Input    : log-magnitude spectrogram
  Loss     : GT + KD + Energy + log-STFT
  Epochs   : 20
  Batches  : 500 train | 15 val

  E01 [000/500] loss=0.0836 (gt=0.078 kd=0.073 energy=0.0

KeyboardInterrupt: 

In [1]:
# ============================================================
# PRECOMPUTE DEMUCS VOCALS — RUN ONCE BEFORE TRAINING
# Saves Demucs vocal predictions to disk
# Training will load from disk instead of running Demucs live
# ============================================================

import os, sys, gc, torch, torchaudio
from pathlib import Path

PROJECT      = Path("C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION")
DATASET      = Path("D:/dataset")
DEMUCS_CACHE = PROJECT / "data" / "demucs_cache"

sys.path.insert(0, str(PROJECT))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"Cache  : {DEMUCS_CACHE}")

# Install demucs if needed
try:
    from demucs.pretrained import get_model
    from demucs.apply import apply_model
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "demucs", "-q"])
    from demucs.pretrained import get_model
    from demucs.apply import apply_model

# Load Demucs once
print("\nLoading Demucs htdemucs...")
demucs       = get_model("htdemucs")
demucs.to(device)
demucs.eval()
for p in demucs.parameters():
    p.requires_grad = False

SOURCES    = demucs.sources
VOCALS_IDX = SOURCES.index("vocals")
print(f"✓ Loaded | sources={SOURCES} | vocals_idx={VOCALS_IDX}")

def process_split(split: str) -> None:
    split_dir = DATASET / split
    out_dir   = DEMUCS_CACHE / split
    out_dir.mkdir(parents=True, exist_ok=True)

    songs = sorted(split_dir.iterdir())
    print(f"\n{split.upper()} — {len(songs)} songs")
    print("-" * 50)

    for song_dir in songs:
        mix_path = song_dir / "mixture.wav"
        if not mix_path.exists():
            continue

        out_path = out_dir / f"{song_dir.name}.wav"

        # Skip if already done
        if out_path.exists():
            print(f"  SKIP (cached) : {song_dir.name}")
            continue

        # Load full song
        audio, sr = torchaudio.load(str(mix_path))
        if sr != 44100:
            audio = torchaudio.functional.resample(audio, sr, 44100)
        if audio.shape[0] == 1:
            audio = audio.repeat(2, 1)
        elif audio.shape[0] > 2:
            audio = audio[:2]

        total_frames = audio.shape[1]

        # Process in 30-second chunks to avoid OOM
        chunk_size   = 44100 * 30
        vocals_parts = []

        for start in range(0, total_frames, chunk_size):
            end   = min(start + chunk_size, total_frames)
            chunk = audio[:, start:end].unsqueeze(0).to(device)

            with torch.no_grad():
                sources = apply_model(
                    demucs, chunk,
                    progress=False, num_workers=0,
                )
            vocals_parts.append(sources[0, VOCALS_IDX].cpu())
            del chunk, sources
            gc.collect()
            torch.cuda.empty_cache()

        vocals_full = torch.cat(vocals_parts, dim=1)
        torchaudio.save(str(out_path), vocals_full, 44100)

        duration = total_frames / 44100
        print(f"  DONE : {song_dir.name}  ({duration:.1f}s)")
        del audio, vocals_full, vocals_parts

# Run both splits
process_split("train")
process_split("test")

# Report
print("\n" + "=" * 50)
print("PRECOMPUTATION COMPLETE")
print("=" * 50)
train_count = len(list((DEMUCS_CACHE / "train").glob("*.wav")))
test_count  = len(list((DEMUCS_CACHE / "test").glob("*.wav")))
print(f"  Train cached : {train_count} songs")
print(f"  Test cached  : {test_count} songs")
print(f"  Cache folder : {DEMUCS_CACHE}")
print("\n the training cell.")

Device : cuda
Cache  : C:\Users\Disha Saini\Documents\ML\AUDIO_SEPARATION\data\demucs_cache

Loading Demucs htdemucs...
✓ Loaded | sources=['drums', 'bass', 'other', 'vocals'] | vocals_idx=3

TRAIN — 100 songs
--------------------------------------------------
  DONE : A Classic Education - NightOwl  (171.4s)
  DONE : Actions - Devil's Words  (196.8s)
  DONE : Actions - One Minute Smile  (163.6s)
  DONE : Actions - South Of The Water  (176.8s)
  DONE : Aimee Norwich - Child  (189.3s)
  DONE : Alexander Ross - Goodbye Bolero  (418.8s)
  DONE : Alexander Ross - Velvet Curtain  (514.5s)
  DONE : Angela Thomas Wade - Milk Cow Blues  (211.1s)
  DONE : ANiMAL - Clinic A  (238.0s)
  DONE : ANiMAL - Easy Tiger  (205.7s)
  DONE : ANiMAL - Rockshow  (165.7s)
  DONE : Atlantis Bound - It Was My Fault For Waiting  (268.2s)
  DONE : Auctioneer - Our Future Faces  (207.9s)
  DONE : AvaLuna - Waterduct  (259.3s)
  DONE : BigTroubles - Phantom  (146.9s)
  DONE : Bill Chudziak - Children Of No-one  (23

LibsndfileError: System error.

In [2]:
# ============================================================
# PRECOMPUTE DEMUCS — FIXED VERSION
# Handles special characters in song names
# Resumes from where it crashed (train is already done)
# ============================================================

import os, sys, gc, torch, torchaudio
from pathlib import Path

PROJECT      = Path("C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION")
DATASET      = Path("D:/dataset")
DEMUCS_CACHE = PROJECT / "data" / "demucs_cache"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from demucs.pretrained import get_model
from demucs.apply import apply_model

print("Loading Demucs...")
demucs       = get_model("htdemucs")
demucs.to(device)
demucs.eval()
for p in demucs.parameters():
    p.requires_grad = False

VOCALS_IDX = demucs.sources.index("vocals")
print(f"✓ Loaded | vocals_idx={VOCALS_IDX}")

def safe_save(audio, path):
    """
    Save audio with fallback for paths with special characters.
    Tries direct save first, then falls back to temp file + rename.
    """
    try:
        torchaudio.save(str(path), audio, 44100)
        return True
    except Exception:
        # Fallback: save to a safe temp path then rename
        safe_path = path.parent / (path.stem.encode("ascii", "ignore").decode() + ".wav")
        try:
            torchaudio.save(str(safe_path), audio, 44100)
            # Rename back to original if safe_path differs
            if safe_path != path:
                safe_path.rename(path)
            return True
        except Exception as e2:
            print(f"    ✗ Could not save {path.name}: {e2}")
            return False

def process_split(split: str):
    split_dir = DATASET / split
    out_dir   = DEMUCS_CACHE / split
    out_dir.mkdir(parents=True, exist_ok=True)

    songs = sorted(split_dir.iterdir())
    done  = 0
    skip  = 0

    print(f"\n{split.upper()} — {len(songs)} songs")
    print("-" * 50)

    for song_dir in songs:
        mix_path = song_dir / "mixture.wav"
        if not mix_path.exists():
            continue

        out_path = out_dir / f"{song_dir.name}.wav"

        if out_path.exists():
            skip += 1
            continue

        # Load full song
        try:
            audio, sr = torchaudio.load(str(mix_path))
        except Exception as e:
            print(f"  ✗ Load failed: {song_dir.name}: {e}")
            continue

        if sr != 44100:
            audio = torchaudio.functional.resample(audio, sr, 44100)
        if audio.shape[0] == 1:
            audio = audio.repeat(2, 1)
        elif audio.shape[0] > 2:
            audio = audio[:2]

        # Process in 30-second chunks
        chunk_size   = 44100 * 30
        total_frames = audio.shape[1]
        vocals_parts = []

        for start in range(0, total_frames, chunk_size):
            end   = min(start + chunk_size, total_frames)
            chunk = audio[:, start:end].unsqueeze(0).to(device)

            with torch.no_grad():
                sources = apply_model(
                    demucs, chunk,
                    progress=False, num_workers=0,
                )
            vocals_parts.append(sources[0, VOCALS_IDX].cpu())
            del chunk, sources
            gc.collect()
            torch.cuda.empty_cache()

        vocals_full = torch.cat(vocals_parts, dim=1)

        if safe_save(vocals_full, out_path):
            done += 1
            print(f"  DONE : {song_dir.name}  ({total_frames/44100:.1f}s)")
        else:
            print(f"  FAIL : {song_dir.name}")

        del audio, vocals_full, vocals_parts

    print(f"\n  Done: {done}  |  Skipped (cached): {skip}")

# Train is already done — only run test
process_split("test")

# Final report
print("\n" + "=" * 50)
train_count = len(list((DEMUCS_CACHE / "train").glob("*.wav")))
test_count  = len(list((DEMUCS_CACHE / "test").glob("*.wav")))
print(f"Train cached : {train_count} / 100")
print(f"Test cached  : {test_count} / 50")

if test_count < 50:
    missing = 50 - test_count
    print(f"⚠ {missing} test songs missing — check for special characters in names")
else:
    print("✓ All songs cached — run training cell now")

Loading Demucs...
✓ Loaded | vocals_idx=3

TEST — 50 songs
--------------------------------------------------
  DONE : Buitraker - Revo X  (275.8s)
    ✗ Could not save Carlos Gonzalez - A Place For Us.wav: System error.
  FAIL : Carlos Gonzalez - A Place For Us
    ✗ Could not save Cristina Vane - So Easy.wav: System error.
  FAIL : Cristina Vane - So Easy
    ✗ Could not save Detsky Sad - Walkie Talkie.wav: System error.
  FAIL : Detsky Sad - Walkie Talkie
    ✗ Could not save Enda Reilly - Cur An Long Ag Seol.wav: System error.
  FAIL : Enda Reilly - Cur An Long Ag Seol
    ✗ Could not save Forkupines - Semantics.wav: System error.
  FAIL : Forkupines - Semantics
  DONE : Georgia Wonder - Siren  (430.4s)
  DONE : Girls Under Glass - We Feel Alright  (317.3s)
  DONE : Hollow Ground - Ill Fate  (141.7s)
  DONE : James Elder & Mark M Thompson - The English Actor  (205.5s)
  DONE : Juliet's Rescue - Heartbeats  (267.7s)
  DONE : Little Chicago's Finest - My Own  (281.6s)
  DONE : Louis 

In [3]:
# ============================================================
# PHASE 5 — TRAINING (run this now)
# All demucs outputs are precomputed on disk
# GPU only used for GSN model — no Demucs during training
# ============================================================

import os, sys, gc, random, time, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

PROJECT      = Path("C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION")
DATASET      = Path("D:/dataset")
DEMUCS_CACHE = PROJECT / "data" / "demucs_cache"

sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

for k in list(sys.modules.keys()):
    if "src" in k: del sys.modules[k]

gc.collect()
torch.cuda.empty_cache()
torch.manual_seed(42)

from src.models.unet import UNetConfig, EncoderBlock, DecoderBlock
from src.models.harmonic_graph import build_harmonic_edges
from src.models.gcn_bottleneck import HarmonicGCNBottleneck
from src.training.losses import compute_si_sdr

device = torch.device("cuda")
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB\n")

# ============================================================
# DATASET
# ============================================================
class VocalDataset(Dataset):
    """
    Returns (mixture, gt_vocals, demucs_vocals) per sample.
    All three loaded from disk — Demucs never runs during training.
    """

    def __init__(self, dataset_path, demucs_cache, split,
                 chunk_duration=4.0, sample_rate=44100):
        self.chunk_n = int(chunk_duration * sample_rate)
        self.sr      = sample_rate
        self.songs   = []

        split_dir  = dataset_path / split
        cache_dir  = demucs_cache / split

        # Load index — maps song name to cache filename
        index_path = cache_dir / "index.json"

        if index_path.exists():
            with open(index_path, "r", encoding="utf-8") as f:
                index = json.load(f)
        else:
            # Fallback: use song name as filename (old format)
            index = {
                d.name: f"{d.name}.wav"
                for d in sorted(split_dir.iterdir())
                if (cache_dir / f"{d.name}.wav").exists()
            }

        for song_dir in sorted(split_dir.iterdir()):
            mix_path    = song_dir / "mixture.wav"
            vocals_path = song_dir / "vocals.wav"
            song_name   = song_dir.name

            # Find cache file — try index first, then direct name
            if song_name in index:
                cache_file = cache_dir / index[song_name]
            else:
                cache_file = cache_dir / f"{song_name}.wav"

            if (mix_path.exists() and
                    vocals_path.exists() and
                    cache_file.exists()):
                self.songs.append({
                    "mix"   : mix_path,
                    "vocals": vocals_path,
                    "demucs": cache_file,
                })

        self.chunks_per_song = 8 if split == "train" else 1
        print(f"  {split}: {len(self.songs)} songs × "
              f"{self.chunks_per_song} = {len(self)} chunks")

    def __len__(self):
        return len(self.songs) * self.chunks_per_song

    def __getitem__(self, idx):
        song  = self.songs[idx // self.chunks_per_song]
        n     = self.chunk_n
        info  = torchaudio.info(str(song["mix"]))
        start = random.randint(0, max(0, info.num_frames - n))
        mix    = self._load(song["mix"],    start, n)
        vocals = self._load(song["vocals"], start, n)
        demucs = self._load(song["demucs"], start, n)
        return mix, vocals, demucs

    def _load(self, path, start, n):
        audio, sr = torchaudio.load(
            str(path), frame_offset=start, num_frames=n
        )
        if sr != self.sr:
            audio = torchaudio.functional.resample(audio, sr, self.sr)
        if audio.shape[0] == 1:
            audio = audio.repeat(2, 1)
        elif audio.shape[0] > 2:
            audio = audio[:2]
        if audio.shape[1] < n:
            audio = F.pad(audio, (0, n - audio.shape[1]))
        return torch.clamp(audio[:, :n].float(), -1.0, 1.0)


# ============================================================
# STFT
# ============================================================
class AudioSTFT(nn.Module):
    def __init__(self, n_fft=2048, hop_length=512):
        super().__init__()
        self.n_fft      = n_fft
        self.hop_length = hop_length
        self.register_buffer("window", torch.hann_window(n_fft))

    def encode(self, audio):
        B, C, T = audio.shape
        flat = audio.reshape(B * C, T)
        spec = torch.stft(flat, self.n_fft, self.hop_length,
                          window=self.window, center=True,
                          return_complex=True)
        _, F, Tf = spec.shape
        spec = spec.reshape(B, C, F, Tf)
        return spec.abs(), torch.angle(spec), T

    def decode(self, magnitude, phase, length=None):
        B, C, F, Tf = magnitude.shape
        cmplx = magnitude * torch.exp(1j * phase)
        flat  = cmplx.reshape(B * C, F, Tf)
        audio = torch.istft(flat, self.n_fft, self.hop_length,
                            window=self.window, center=True, length=length)
        return audio.reshape(B, C, -1)


# ============================================================
# LOSS
# ============================================================
class DistillationLoss(nn.Module):
    """
    0.4 × L1 vs ground truth     — direct supervision
    0.3 × L1 vs Demucs output    — perceptual clarity
    0.1 × energy constraint      — prevents silence
    0.2 × log-STFT (2 scales)    — reduces muffling
    """
    def __init__(self):
        super().__init__()
        self.register_buffer("win_1024", torch.hann_window(1024))
        self.register_buffer("win_2048", torch.hann_window(2048))

    def _log_stft_l1(self, pred, target, n_fft, hop, win):
        p   = pred.reshape(-1, pred.shape[-1])
        t   = target.reshape(-1, target.shape[-1])
        p_m = torch.stft(p, n_fft, hop, n_fft, win,
                         return_complex=True, center=True).abs()
        t_m = torch.stft(t, n_fft, hop, n_fft, win,
                         return_complex=True, center=True).abs()
        return F.l1_loss(torch.log1p(p_m), torch.log1p(t_m))

    def forward(self, pred, gt_vocals, demucs_vocals):
        l_gt     = F.l1_loss(pred, gt_vocals)
        l_kd     = F.l1_loss(pred, demucs_vocals.detach())
        l_energy = (pred.abs().mean() - gt_vocals.abs().mean()).abs()
        l_stft   = (
            self._log_stft_l1(pred, gt_vocals, 1024, 256, self.win_1024) +
            self._log_stft_l1(pred, gt_vocals, 2048, 512, self.win_2048)
        ) / 2.0
        total = 0.4*l_gt + 0.3*l_kd + 0.1*l_energy + 0.2*l_stft
        return total, {
            "gt": l_gt.item(), "kd": l_kd.item(),
            "energy": l_energy.item(), "stft": l_stft.item(),
        }


# ============================================================
# MODEL
# ============================================================
class ImprovedGSN(nn.Module):
    """
    GSN with:
      - log-magnitude mono input
      - magnitude mask output (sigmoid)
      - harmonic GCN bottleneck
    """
    def __init__(self, n_freq_bins=1025, base_channels=32,
                 depth=4, dropout=0.1):
        super().__init__()
        ch = [base_channels * (2 ** i) for i in range(depth)]

        self.encoders = nn.ModuleList()
        self.encoders.append(EncoderBlock(1, ch[0], dropout, True))
        for i in range(1, depth):
            self.encoders.append(EncoderBlock(ch[i-1], ch[i], dropout, True))

        bn_freq  = max(n_freq_bins // (2 ** depth), 4)
        bn_nfft  = max(2048 // (2 ** depth), 64)
        edge_idx = build_harmonic_edges(bn_freq, 44100, bn_nfft, 5)
        self.bottleneck = HarmonicGCNBottleneck(ch[-1], edge_idx, dropout)

        self.decoders = nn.ModuleList()
        for i in range(depth - 1, 0, -1):
            self.decoders.append(DecoderBlock(ch[i], ch[i], ch[i-1], dropout))
        self.decoders.append(DecoderBlock(ch[0], ch[0], ch[0], dropout))

        self.mask_head = nn.Sequential(
            nn.Conv2d(ch[0], 1, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, log_mag_mono):
        """log_mag_mono [B,1,F,T] -> mask [B,1,F,T]"""
        x, skips = log_mag_mono, []
        for enc in self.encoders:
            x, skip = enc(x)
            skips.append(skip)
        x = self.bottleneck(x)
        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)
        return self.mask_head(x)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ============================================================
# BUILD
# ============================================================
stft    = AudioSTFT().to(device)
model   = ImprovedGSN().to(device)
loss_fn = DistillationLoss().to(device)

print(f"Model parameters: {model.count_parameters():,}")

# Load Phase 2 weights
p2_path = PROJECT / "checkpoints" / "phase2" / "phase2_final_3.12dB.pt"
if p2_path.exists():
    p2_state = torch.load(p2_path, map_location=device)["model_state"]
    m_state  = model.state_dict()
    loaded   = sum(
        1 for k, v in p2_state.items()
        if k in m_state and m_state[k].shape == v.shape
        and not m_state.update({k: v})
    )
    model.load_state_dict(m_state)
    print(f"Loaded {loaded} Phase 2 tensors")

optimizer = torch.optim.Adam(
    model.parameters(), lr=3e-4, weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=3, factor=0.5, min_lr=1e-6,
)

# Datasets
print("\nBuilding datasets...")
train_ds = VocalDataset(DATASET, DEMUCS_CACHE, "train")
val_ds   = VocalDataset(DATASET, DEMUCS_CACHE, "test")

train_loader = DataLoader(
    train_ds, batch_size=4, shuffle=True, num_workers=0, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=4, shuffle=False, num_workers=0,
)

# Save dirs
save_dir      = PROJECT / "checkpoints" / "phase5"
audio_out_dir = PROJECT / "outputs" / "audio" / "phase5_samples"
save_dir.mkdir(parents=True, exist_ok=True)
audio_out_dir.mkdir(parents=True, exist_ok=True)
best_path = save_dir / "best_vocal_model.pt"

def save_ckpt(epoch, val_sdr, history):
    torch.save({
        "epoch"      : epoch,
        "model_state": model.state_dict(),
        "val_si_sdr" : val_sdr,
        "history"    : history,
    }, best_path)

def save_audio_sample(pred, gt, mix, epoch):
    def norm(t):
        mx = t.abs().max()
        return t / mx * 0.9 if mx > 0 else t
    for audio, tag in [(pred, "pred"), (gt, "gt"), (mix, "mix")]:
        torchaudio.save(
            str(audio_out_dir / f"ep{epoch:02d}_{tag}.wav"),
            norm(audio[0].cpu()), 44100,
        )
    print(f"  Audio saved → phase5_samples/ep{epoch:02d}_*.wav")

def cpu_sdr(pred, target):
    return compute_si_sdr(pred.detach().cpu(), target.detach().cpu())

# ============================================================
# SANITY CHECK
# ============================================================
print("\nSanity check...")
with torch.no_grad():
    s_mix, s_gt, s_dem = next(iter(train_loader))
    s_mix = s_mix.to(device)
    s_gt  = s_gt.to(device)
    s_dem = s_dem.to(device)

    mag, phase, T = stft.encode(s_mix)
    log_mag       = torch.log1p(mag)
    mono          = log_mag.mean(dim=1, keepdim=True)
    mask          = model(mono).repeat(1, 2, 1, 1)
    pred_mag      = mag * mask
    pred_aud      = stft.decode(pred_mag, phase, length=T)
    loss, comps   = loss_fn(pred_aud, s_gt, s_dem)
    sdr           = cpu_sdr(pred_aud, s_gt)

print(f"  Batch shape : {s_mix.shape}")
print(f"  Mask range  : [{mask.min():.3f}, {mask.max():.3f}]")
print(f"  Loss        : {loss.item():.4f}")
print(f"  SDR         : {sdr:+.2f} dB")
assert loss.item() < 10.0, "Loss too high — check dataset"
print("  ✓ Passed\n")

saved_mix = s_mix.detach()
saved_gt  = s_gt.detach()
del s_mix, s_gt, s_dem, mag, phase, log_mag, mono, mask, pred_mag, pred_aud
gc.collect()
torch.cuda.empty_cache()

# ============================================================
# TRAINING
# ============================================================
MAX_EPOCHS  = 10
LOG_EVERY   = 20
VAL_BATCHES = 12

print("=" * 60)
print("PHASE 5 — GSN + KNOWLEDGE DISTILLATION")
print(f"  Epochs   : {MAX_EPOCHS}")
print(f"  Batches  : {len(train_loader)} train | {VAL_BATCHES} val")
print(f"  Demucs   : loaded from disk (GPU-safe)")
print("=" * 60 + "\n")

best_val_sdr = float("-inf")
history      = {"train_loss": [], "val_sdr": [], "lr": []}

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    t_loss_sum  = 0.0
    t_sdr_sum   = 0.0
    t_sdr_count = 0
    t0          = time.time()

    for step, (mix_aud, gt_vocals, demucs_vocals) in enumerate(train_loader):
        mix_aud       = mix_aud.to(device)
        gt_vocals     = gt_vocals.to(device)
        demucs_vocals = demucs_vocals.to(device)

        mag, phase, T = stft.encode(mix_aud)
        log_mag       = torch.log1p(mag)
        mono          = log_mag.mean(dim=1, keepdim=True)
        mask          = model(mono).repeat(1, 2, 1, 1)
        pred_mag      = mag * mask
        pred_aud      = stft.decode(pred_mag, phase, length=T)

        loss, comps = loss_fn(pred_aud, gt_vocals, demucs_vocals)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        t_loss_sum += loss.item()

        if step % LOG_EVERY == 0:
            sdr = cpu_sdr(pred_aud, gt_vocals)
            t_sdr_sum   += sdr
            t_sdr_count += 1
            eta = (time.time()-t0)/(step+1)*(len(train_loader)-step-1)
            print(
                f"  E{epoch:02d} [{step:03d}/{len(train_loader)}] "
                f"loss={loss.item():.4f} "
                f"gt={comps['gt']:.3f} "
                f"kd={comps['kd']:.3f} "
                f"stft={comps['stft']:.3f} "
                f"sdr={sdr:+.2f}dB "
                f"ETA={eta:.0f}s"
            )

        del mix_aud, gt_vocals, demucs_vocals
        del mag, phase, log_mag, mono, mask, pred_mag, pred_aud, loss

    avg_t_loss = t_loss_sum / len(train_loader)
    avg_t_sdr  = t_sdr_sum  / max(1, t_sdr_count)

    # Validate
    model.eval()
    v_sdr_sum = 0.0
    v_count   = 0
    last_pred = last_gt = last_mix = None

    with torch.no_grad():
        for vi, (mix_aud, gt_vocals, _) in enumerate(val_loader):
            if vi >= VAL_BATCHES:
                break
            mix_aud   = mix_aud.to(device)
            gt_vocals = gt_vocals.to(device)

            mag, phase, T = stft.encode(mix_aud)
            log_mag       = torch.log1p(mag)
            mono          = log_mag.mean(dim=1, keepdim=True)
            mask          = model(mono).repeat(1, 2, 1, 1)
            pred_mag      = mag * mask
            pred_aud      = stft.decode(pred_mag, phase, length=T)

            sdr        = cpu_sdr(pred_aud, gt_vocals)
            v_sdr_sum += sdr
            v_count   += 1

            if vi == 0:
                last_pred = pred_aud.detach()
                last_gt   = gt_vocals.detach()
                last_mix  = mix_aud.detach()

            del mix_aud, gt_vocals, mag, phase, log_mag, mono, mask, pred_mag, pred_aud

    avg_val_sdr = v_sdr_sum / max(1, v_count)
    lr          = optimizer.param_groups[0]["lr"]
    scheduler.step(avg_val_sdr)
    elapsed     = time.time() - t0

    history["train_loss"].append(avg_t_loss)
    history["val_sdr"].append(avg_val_sdr)
    history["lr"].append(lr)

    print(f"\n{'='*60}")
    print(f"EPOCH {epoch}/{MAX_EPOCHS}  ({elapsed:.0f}s)  lr={lr:.1e}")
    print(f"  Train: loss={avg_t_loss:.4f}  sdr={avg_t_sdr:+.2f} dB")
    print(f"  Val  : sdr={avg_val_sdr:+.2f} dB")
    print(f"{'='*60}")

    # Save audio every 2 epochs
    if epoch % 2 == 0 and last_pred is not None:
        save_audio_sample(last_pred, last_gt, last_mix, epoch)

    if avg_val_sdr > best_val_sdr:
        best_val_sdr = avg_val_sdr
        save_ckpt(epoch, avg_val_sdr, history)
        print(f"★ New best: {best_val_sdr:+.2f} dB  (saved)\n")
    else:
        print()

print("=" * 60)
print("PHASE 5 COMPLETE")
print("=" * 60)
print(f"  Phase 2 baseline  : +3.12 dB")
print(f"  Phase 3 CLAP      : +3.80 dB")
print(f"  Phase 5 result    : {best_val_sdr:+.2f} dB")
print()
print("Next: Run the Hybrid System (Phase 6)")

GPU : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.4 GB

Model parameters: 2,301,634
Loaded 106 Phase 2 tensors

Building datasets...
  train: 100 songs × 8 = 800 chunks
  test: 50 songs × 1 = 50 chunks

Sanity check...
  Batch shape : torch.Size([4, 2, 176400])
  Mask range  : [0.066, 0.424]
  Loss        : 0.0418
  SDR         : -22.11 dB
  ✓ Passed

PHASE 5 — GSN + KNOWLEDGE DISTILLATION
  Epochs   : 10
  Batches  : 200 train | 12 val
  Demucs   : loaded from disk (GPU-safe)

  E01 [000/200] loss=0.0371 gt=0.018 kd=0.018 stft=0.115 sdr=-58.53dB ETA=440s
  E01 [020/200] loss=0.0355 gt=0.029 kd=0.027 stft=0.071 sdr=+3.08dB ETA=244s
  E01 [040/200] loss=0.0519 gt=0.033 kd=0.028 stft=0.146 sdr=-29.56dB ETA=207s
  E01 [060/200] loss=0.0452 gt=0.034 kd=0.030 stft=0.100 sdr=+2.66dB ETA=178s
  E01 [080/200] loss=0.0446 gt=0.037 kd=0.030 stft=0.100 sdr=-10.73dB ETA=151s
  E01 [100/200] loss=0.0373 gt=0.029 kd=0.025 stft=0.084 sdr=+4.21dB ETA=125s
  E01 [120/200] loss=0.0364 gt=0.025 kd=0.0

KeyboardInterrupt: 

In [2]:
# Quick diagnostic — check what checkpoints exist and what they contain

import torch
from pathlib import Path

PROJECT = Path("C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION")

print("ALL CHECKPOINTS:")
print("=" * 60)

checkpoint_dirs = [
    "checkpoints/phase1",
    "checkpoints/phase2", 
    "checkpoints/phase3",
    "checkpoints/phase4",
    "checkpoints/phase4a",
    "checkpoints/phase5",
    "checkpoints/phase3_production",
    "checkpoints/phase3_final",
    "checkpoints/gsn_improved",
]

best_sdr     = float("-inf")
best_ckpt    = None

for ckpt_dir in checkpoint_dirs:
    full_dir = PROJECT / ckpt_dir
    if not full_dir.exists():
        continue
    
    pt_files = list(full_dir.glob("*.pt"))
    if not pt_files:
        continue
    
    print(f"\n{ckpt_dir}/")
    
    for pt_file in sorted(pt_files):
        try:
            ckpt     = torch.load(pt_file, map_location="cpu")
            epoch    = ckpt.get("epoch", "?")
            val_sdr  = ckpt.get("val_si_sdr", None)
            val_loss = ckpt.get("val_loss", None)
            keys     = list(ckpt.get("model_state", {}).keys())
            n_keys   = len(keys)
            size_mb  = pt_file.stat().st_size / 1e6

            sdr_str = f"{val_sdr:+.2f} dB" if val_sdr is not None else "N/A"
            
            print(f"  {pt_file.name:<35} epoch={epoch}  sdr={sdr_str}  keys={n_keys}  {size_mb:.1f}MB")

            if val_sdr is not None and val_sdr > best_sdr:
                best_sdr  = val_sdr
                best_ckpt = pt_file

        except Exception as e:
            print(f"  {pt_file.name:<35} ERROR: {e}")

print()
print("=" * 60)
print(f"BEST CHECKPOINT OVERALL:")
print(f"  File : {best_ckpt}")
print(f"  SDR  : {best_sdr:+.2f} dB")
print()
print("=" * 60)
print("WHAT THE CURRENT TRAINING CELL LOADS:")
print(f"  checkpoints/phase2/phase2_final_3.12dB.pt")
print()
print("IS THAT THE BEST? Check above.")
print("If a better checkpoint exists, we should load that instead.")

ALL CHECKPOINTS:

checkpoints/phase1/
  best_model.pt                       epoch=33  sdr=+3.22 dB  keys=110  40.3MB
  phase1_final_3.22dB.pt              epoch=33  sdr=+3.22 dB  keys=110  40.3MB

checkpoints/phase2/
  best_model.pt                       epoch=5  sdr=+3.12 dB  keys=108  27.7MB
  phase2_final_3.12dB.pt              epoch=5  sdr=+3.12 dB  keys=108  27.7MB

checkpoints/phase3/
  best_model.pt                       epoch=3  sdr=+2.43 dB  keys=591  647.1MB
  best_model_clean.pt                 epoch=18  sdr=+3.80 dB  keys=114  32.5MB
  phase3_final_3.80dB.pt              epoch=18  sdr=+3.80 dB  keys=114  32.5MB

checkpoints/phase4/
  best_model.pt                       epoch=1  sdr=-11.38 dB  keys=116  10.8MB

checkpoints/phase4a/
  best_model.pt                       epoch=6  sdr=-6.17 dB  keys=115  9.8MB

checkpoints/phase5/
  best_vocal_model.pt                 epoch=3  sdr=-6.55 dB  keys=108  9.3MB

checkpoints/phase3_production/
  best_model.pt                       ep

In [3]:
# ============================================================
# PHASE 5 — COMPLETE FINAL TRAINING CELL
# Loads best available checkpoint (Phase 3 +3.80 dB)
# Uses SI-SDR loss so metric and training objective match
# ============================================================

import os, sys, gc, random, time, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

PROJECT      = Path("C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION")
DATASET      = Path("D:/dataset")
DEMUCS_CACHE = PROJECT / "data" / "demucs_cache"

sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

for k in list(sys.modules.keys()):
    if "src" in k: del sys.modules[k]

gc.collect()
torch.cuda.empty_cache()
torch.manual_seed(42)

from src.models.unet import UNetConfig, EncoderBlock, DecoderBlock
from src.models.harmonic_graph import build_harmonic_edges
from src.models.gcn_bottleneck import HarmonicGCNBottleneck
from src.training.losses import compute_si_sdr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB\n")

# ============================================================
# DATASET
# ============================================================
class VocalDataset(Dataset):
    def __init__(self, dataset_path, demucs_cache, split,
                 chunk_duration=4.0, sample_rate=44100):
        self.chunk_n = int(chunk_duration * sample_rate)
        self.sr      = sample_rate
        self.songs   = []

        split_dir  = dataset_path / split
        cache_dir  = demucs_cache / split
        index_path = cache_dir / "index.json"

        index = {}
        if index_path.exists():
            with open(index_path, "r", encoding="utf-8") as f:
                index = json.load(f)

        for song_dir in sorted(split_dir.iterdir()):
            mix_path    = song_dir / "mixture.wav"
            vocals_path = song_dir / "vocals.wav"
            song_name   = song_dir.name

            if song_name in index:
                cache_file = cache_dir / index[song_name]
            else:
                cache_file = cache_dir / f"{song_name}.wav"

            if (mix_path.exists() and
                    vocals_path.exists() and
                    cache_file.exists()):
                self.songs.append({
                    "mix"   : mix_path,
                    "vocals": vocals_path,
                    "demucs": cache_file,
                })

        self.chunks_per_song = 8 if split == "train" else 1
        print(f"  {split}: {len(self.songs)} songs × "
              f"{self.chunks_per_song} = {len(self)} chunks")

        if len(self.songs) == 0:
            raise ValueError(f"No songs found for {split}. Check demucs cache.")

    def __len__(self):
        return len(self.songs) * self.chunks_per_song

    def __getitem__(self, idx):
        song  = self.songs[idx // self.chunks_per_song]
        n     = self.chunk_n
        info  = torchaudio.info(str(song["mix"]))
        start = random.randint(0, max(0, info.num_frames - n))
        return (
            self._load(song["mix"],    start, n),
            self._load(song["vocals"], start, n),
            self._load(song["demucs"], start, n),
        )

    def _load(self, path, start, n):
        audio, sr = torchaudio.load(str(path), frame_offset=start, num_frames=n)
        if sr != self.sr:
            audio = torchaudio.functional.resample(audio, sr, self.sr)
        if audio.shape[0] == 1:
            audio = audio.repeat(2, 1)
        elif audio.shape[0] > 2:
            audio = audio[:2]
        if audio.shape[1] < n:
            audio = F.pad(audio, (0, n - audio.shape[1]))
        return torch.clamp(audio[:, :n].float(), -1.0, 1.0)


# ============================================================
# STFT
# ============================================================
class AudioSTFT(nn.Module):
    def __init__(self, n_fft=2048, hop_length=512):
        super().__init__()
        self.n_fft      = n_fft
        self.hop_length = hop_length
        self.register_buffer("window", torch.hann_window(n_fft))

    def encode(self, audio):
        B, C, T = audio.shape
        flat = audio.reshape(B * C, T)
        spec = torch.stft(flat, self.n_fft, self.hop_length,
                          window=self.window, center=True,
                          return_complex=True)
        _, F, Tf = spec.shape
        spec = spec.reshape(B, C, F, Tf)
        return spec.abs(), torch.angle(spec), T

    def decode(self, magnitude, phase, length=None):
        B, C, F, Tf = magnitude.shape
        cmplx = magnitude * torch.exp(1j * phase)
        flat  = cmplx.reshape(B * C, F, Tf)
        audio = torch.istft(flat, self.n_fft, self.hop_length,
                            window=self.window, center=True, length=length)
        return audio.reshape(B, C, -1)


# ============================================================
# LOSS
# ============================================================
class FinalLoss(nn.Module):
    """
    0.7 × negative SI-SNR  — directly optimizes what we measure
    0.2 × log-STFT L1      — reduces muffling
    0.1 × KD from Demucs   — soft perceptual hint
    """
    def __init__(self):
        super().__init__()
        self.register_buffer("win_2048", torch.hann_window(2048))

    def neg_sisnr(self, pred, target, eps=1e-8):
        p = pred.reshape(pred.shape[0], -1)
        t = target.reshape(target.shape[0], -1)
        p = p - p.mean(dim=1, keepdim=True)
        t = t - t.mean(dim=1, keepdim=True)
        dot   = (p * t).sum(dim=1, keepdim=True)
        t_pow = (t * t).sum(dim=1, keepdim=True) + eps
        s_tgt = (dot / t_pow) * t
        noise = p - s_tgt
        sisnr = 10 * torch.log10(
            (s_tgt * s_tgt).sum(dim=1) /
            ((noise * noise).sum(dim=1) + eps) + eps
        )
        return -torch.clamp(sisnr, min=-30.0).mean()

    def log_stft_l1(self, pred, target):
        p   = pred.reshape(-1, pred.shape[-1])
        t   = target.reshape(-1, target.shape[-1])
        p_m = torch.stft(p, 2048, 512, 2048, self.win_2048,
                         return_complex=True, center=True).abs()
        t_m = torch.stft(t, 2048, 512, 2048, self.win_2048,
                         return_complex=True, center=True).abs()
        return F.l1_loss(torch.log1p(p_m), torch.log1p(t_m))

    def forward(self, pred, gt_vocals, demucs_vocals):
        l_sisnr = self.neg_sisnr(pred, gt_vocals)
        l_stft  = self.log_stft_l1(pred, gt_vocals)
        l_kd    = F.l1_loss(pred, demucs_vocals.detach())
        total   = 0.7 * l_sisnr + 0.2 * l_stft + 0.1 * l_kd
        return total, {
            "sisnr": l_sisnr.item(),
            "stft" : l_stft.item(),
            "kd"   : l_kd.item(),
        }


# ============================================================
# MODEL
# ============================================================
class ImprovedGSN(nn.Module):
    def __init__(self, n_freq_bins=1025, base_channels=32,
                 depth=4, dropout=0.1):
        super().__init__()
        ch = [base_channels * (2 ** i) for i in range(depth)]

        self.encoders = nn.ModuleList()
        self.encoders.append(EncoderBlock(1, ch[0], dropout, True))
        for i in range(1, depth):
            self.encoders.append(
                EncoderBlock(ch[i-1], ch[i], dropout, True)
            )

        bn_freq  = max(n_freq_bins // (2 ** depth), 4)
        bn_nfft  = max(2048 // (2 ** depth), 64)
        edge_idx = build_harmonic_edges(bn_freq, 44100, bn_nfft, 5)
        self.bottleneck = HarmonicGCNBottleneck(ch[-1], edge_idx, dropout)

        self.decoders = nn.ModuleList()
        for i in range(depth - 1, 0, -1):
            self.decoders.append(
                DecoderBlock(ch[i], ch[i], ch[i-1], dropout)
            )
        self.decoders.append(
            DecoderBlock(ch[0], ch[0], ch[0], dropout)
        )

        self.mask_conv = nn.Conv2d(ch[0], 1, kernel_size=1)

    def forward(self, log_mag_mono):
        x, skips = log_mag_mono, []
        for enc in self.encoders:
            x, skip = enc(x)
            skips.append(skip)
        x = self.bottleneck(x)
        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)
        mask = torch.sigmoid(self.mask_conv(x)) * 1.5
        return torch.clamp(mask, 0.0, 1.0)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ============================================================
# BUILD
# ============================================================
stft    = AudioSTFT().to(device)
model   = ImprovedGSN().to(device)
loss_fn = FinalLoss().to(device)

print(f"Model parameters: {model.count_parameters():,}")

# ── Load best available checkpoint ───────────────────────────
# Priority: Phase 3 (+3.80) > Phase 1 (+3.22) > Phase 2 (+3.12)
# We load matching layers only — mismatched layers are skipped
checkpoints_to_try = [
    (PROJECT / "checkpoints" / "phase3" / "best_model_clean.pt",    "+3.80 dB"),
    (PROJECT / "checkpoints" / "phase1" / "phase1_final_3.22dB.pt", "+3.22 dB"),
    (PROJECT / "checkpoints" / "phase2" / "phase2_final_3.12dB.pt", "+3.12 dB"),
]

loaded_from = None
for ckpt_path, label in checkpoints_to_try:
    if not ckpt_path.exists():
        continue
    try:
        saved_state = torch.load(ckpt_path, map_location=device)["model_state"]
        m_state     = model.state_dict()
        loaded = skipped = 0
        for k, v in saved_state.items():
            if k in m_state and m_state[k].shape == v.shape:
                m_state[k] = v
                loaded += 1
            else:
                skipped += 1
        model.load_state_dict(m_state)
        loaded_from = ckpt_path
        print(f"✓ Loaded : {ckpt_path.name}  ({label})")
        print(f"  Layers matched  : {loaded}")
        print(f"  Layers skipped  : {skipped}  (expected — arch difference)")
        break
    except Exception as e:
        print(f"✗ Could not load {ckpt_path.name}: {e}")

if loaded_from is None:
    print("No checkpoint loaded — starting from random init")

# ── Optimizer ─────────────────────────────────────────────────
optimizer = torch.optim.Adam(
    model.parameters(), lr=3e-4, weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=3, factor=0.5, min_lr=1e-6,
)

# ── Datasets ──────────────────────────────────────────────────
print("\nBuilding datasets...")
train_ds = VocalDataset(DATASET, DEMUCS_CACHE, "train")
val_ds   = VocalDataset(DATASET, DEMUCS_CACHE, "test")

train_loader = DataLoader(
    train_ds, batch_size=4, shuffle=True, num_workers=0, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=4, shuffle=False, num_workers=0,
)

# ── Save dirs ─────────────────────────────────────────────────
save_dir      = PROJECT / "checkpoints" / "phase5"
audio_out_dir = PROJECT / "outputs" / "audio" / "phase5_samples"
save_dir.mkdir(parents=True, exist_ok=True)
audio_out_dir.mkdir(parents=True, exist_ok=True)
best_path = save_dir / "best_vocal_model.pt"

def save_ckpt(epoch, val_sdr, history):
    torch.save({
        "epoch"      : epoch,
        "model_state": model.state_dict(),
        "val_si_sdr" : val_sdr,
        "history"    : history,
    }, best_path)

def save_audio_sample(pred, gt, mix, epoch):
    def norm(t):
        t  = t[0].cpu().float()
        mx = t.abs().max()
        t  = t / mx * 0.9 if mx > 0 else t
        return torch.clamp(t, -1.0, 1.0)
    for audio, tag in [(pred, "pred"), (gt, "gt"), (mix, "mix")]:
        torchaudio.save(
            str(audio_out_dir / f"ep{epoch:02d}_{tag}.wav"),
            norm(audio), 44100,
        )
    print(f"  Audio saved → phase5_samples/ep{epoch:02d}_*.wav")

def cpu_sdr(pred, target):
    return compute_si_sdr(pred.detach().cpu(), target.detach().cpu())

# ============================================================
# STARTING SDR CHECK
# ============================================================
print("\nChecking starting performance before training...")
model.eval()
with torch.no_grad():
    check_mix, check_gt, _ = next(iter(val_loader))
    check_mix = check_mix.to(device)
    check_gt  = check_gt.to(device)

    mag, phase, T = stft.encode(check_mix)
    mono          = torch.log1p(mag).mean(dim=1, keepdim=True)
    mask          = model(mono).repeat(1, 2, 1, 1)
    pred_mag      = mag * mask
    pred_aud      = stft.decode(pred_mag, phase, length=T)
    starting_sdr  = cpu_sdr(pred_aud, check_gt)
    mask_range    = (mask.min().item(), mask.max().item())

print(f"  Starting val SDR : {starting_sdr:+.2f} dB")
print(f"  Mask range       : [{mask_range[0]:.3f}, {mask_range[1]:.3f}]")

if starting_sdr > 2.0:
    print("  ✓ Excellent start — already above +2 dB")
elif starting_sdr > 0.0:
    print("  ✓ Good start — above 0 dB, will improve fast")
elif starting_sdr > -5.0:
    print("  ⚠ Moderate start — will reach positive by epoch 2")
else:
    print("  ⚠ Low start — weights did not transfer well to new arch")
    print("    Training will still work, just takes more epochs")

del check_mix, check_gt, mag, phase, mono, mask, pred_mag, pred_aud
gc.collect()
torch.cuda.empty_cache()

# ============================================================
# SANITY CHECK
# ============================================================
print("\nSanity check on training batch...")
with torch.no_grad():
    s_mix, s_gt, s_dem = next(iter(train_loader))
    s_mix = s_mix.to(device)
    s_gt  = s_gt.to(device)
    s_dem = s_dem.to(device)

    mag, phase, T = stft.encode(s_mix)
    mono          = torch.log1p(mag).mean(dim=1, keepdim=True)
    mask          = model(mono).repeat(1, 2, 1, 1)
    pred_mag      = mag * mask
    pred_aud      = stft.decode(pred_mag, phase, length=T)
    loss, comps   = loss_fn(pred_aud, s_gt, s_dem)

print(f"  Loss : {loss.item():.4f}  (should be < 10)")
print(f"  SISNR: {comps['sisnr']:+.2f}")
print(f"  STFT : {comps['stft']:.4f}")
assert not torch.isnan(loss), "Loss is NaN — check dataset"
assert loss.item() < 20.0,    f"Loss too high: {loss.item()}"
print("  ✓ Sanity check passed\n")

del s_mix, s_gt, s_dem, mag, phase, mono, mask, pred_mag, pred_aud
gc.collect()
torch.cuda.empty_cache()

# ============================================================
# TRAINING
# ============================================================
MAX_EPOCHS  = 10
LOG_EVERY   = 20
VAL_BATCHES = 15
MAX_LOSS    = 10.0

print("=" * 60)
print("PHASE 5 — FINAL TRAINING")
print(f"  Starting from    : {loaded_from.name if loaded_from else 'random'}")
print(f"  Starting SDR     : {starting_sdr:+.2f} dB")
print(f"  Loss             : 0.7×SI-SNR + 0.2×log-STFT + 0.1×KD")
print(f"  Epochs           : {MAX_EPOCHS}")
print(f"  LR               : {optimizer.param_groups[0]['lr']:.0e}")
print("=" * 60 + "\n")

best_val_sdr = float("-inf")
history      = {"train_loss": [], "val_sdr": [], "lr": []}

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    t_loss_sum  = 0.0
    t_sdr_sum   = 0.0
    t_sdr_count = 0
    skipped     = 0
    t0          = time.time()

    for step, (mix_aud, gt_vocals, demucs_vocals) in enumerate(train_loader):
        mix_aud       = mix_aud.to(device)
        gt_vocals     = gt_vocals.to(device)
        demucs_vocals = demucs_vocals.to(device)

        mag, phase, T = stft.encode(mix_aud)
        mono          = torch.log1p(mag).mean(dim=1, keepdim=True)
        mask          = model(mono).repeat(1, 2, 1, 1)
        pred_mag      = mag * mask
        pred_aud      = stft.decode(pred_mag, phase, length=T)

        loss, comps = loss_fn(pred_aud, gt_vocals, demucs_vocals)

        if torch.isnan(loss) or torch.isinf(loss) or loss.item() > MAX_LOSS:
            skipped += 1
            optimizer.zero_grad()
            del mix_aud, gt_vocals, demucs_vocals
            del mag, phase, mono, mask, pred_mag, pred_aud, loss
            continue

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        t_loss_sum += loss.item()

        if step % LOG_EVERY == 0:
            sdr = cpu_sdr(pred_aud, gt_vocals)
            t_sdr_sum   += sdr
            t_sdr_count += 1
            eta = (time.time()-t0)/(step+1)*(len(train_loader)-step-1)
            print(
                f"  E{epoch:02d} [{step:03d}/{len(train_loader)}] "
                f"loss={loss.item():.3f} "
                f"sisnr={comps['sisnr']:+.2f} "
                f"stft={comps['stft']:.3f} "
                f"sdr={sdr:+.2f}dB "
                f"skip={skipped} "
                f"ETA={eta:.0f}s"
            )

        del mix_aud, gt_vocals, demucs_vocals
        del mag, phase, mono, mask, pred_mag, pred_aud, loss

    valid      = len(train_loader) - skipped
    avg_t_loss = t_loss_sum / max(1, valid)
    avg_t_sdr  = t_sdr_sum  / max(1, t_sdr_count)

    # ── Validate ──────────────────────────────────────────────
    model.eval()
    v_sdr_sum = 0.0
    v_count   = 0
    last_pred = last_gt = last_mix = None

    with torch.no_grad():
        for vi, (mix_aud, gt_vocals, _) in enumerate(val_loader):
            if vi >= VAL_BATCHES:
                break
            mix_aud   = mix_aud.to(device)
            gt_vocals = gt_vocals.to(device)

            mag, phase, T = stft.encode(mix_aud)
            mono          = torch.log1p(mag).mean(dim=1, keepdim=True)
            mask          = model(mono).repeat(1, 2, 1, 1)
            pred_mag      = mag * mask
            pred_aud      = stft.decode(pred_mag, phase, length=T)

            sdr        = cpu_sdr(pred_aud, gt_vocals)
            v_sdr_sum += sdr
            v_count   += 1

            if vi == 0:
                last_pred = pred_aud.detach()
                last_gt   = gt_vocals.detach()
                last_mix  = mix_aud.detach()

            del mix_aud, gt_vocals, mag, phase, mono, mask, pred_mag, pred_aud

    avg_val_sdr = v_sdr_sum / max(1, v_count)
    lr          = optimizer.param_groups[0]["lr"]
    scheduler.step(avg_val_sdr)
    elapsed     = time.time() - t0

    history["train_loss"].append(avg_t_loss)
    history["val_sdr"].append(avg_val_sdr)
    history["lr"].append(lr)

    print(f"\n{'='*60}")
    print(f"EPOCH {epoch}/{MAX_EPOCHS}  ({elapsed:.0f}s)  "
          f"lr={lr:.1e}  skipped={skipped}")
    print(f"  Train : loss={avg_t_loss:.4f}  sdr={avg_t_sdr:+.2f} dB")
    print(f"  Val   : sdr={avg_val_sdr:+.2f} dB")
    print(f"{'='*60}")

    if epoch % 2 == 0 and last_pred is not None:
        save_audio_sample(last_pred, last_gt, last_mix, epoch)

    if avg_val_sdr > best_val_sdr:
        best_val_sdr = avg_val_sdr
        save_ckpt(epoch, avg_val_sdr, history)
        print(f"★ New best: {best_val_sdr:+.2f} dB  (saved)\n")
    else:
        print()

# ── Final ─────────────────────────────────────────────────────
print("=" * 60)
print("PHASE 5 COMPLETE")
print("=" * 60)
print(f"  Started from     : {starting_sdr:+.2f} dB")
print(f"  Phase 3 baseline : +3.80 dB")
print(f"  Phase 5 result   : {best_val_sdr:+.2f} dB")
change = best_val_sdr - starting_sdr
print(f"  Change           : {change:+.2f} dB")
print()
if best_val_sdr >= 3.80:
    print("✓ Phase 5 improved over Phase 3 — distillation worked!")
elif best_val_sdr >= 3.0:
    print("✓ Phase 5 result is solid — ready for Phase 6")
else:
    print("Training is still progressing — run more epochs if needed")
print()
print("Next: Phase 6 — Hybrid System (GSN + Demucs)")

Device : cuda
GPU    : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM   : 6.4 GB

Model parameters: 2,301,634
✓ Loaded : best_model_clean.pt  (+3.80 dB)
  Layers matched  : 106
  Layers skipped  : 8  (expected — arch difference)

Building datasets...
  train: 100 songs × 8 = 800 chunks
  test: 50 songs × 1 = 50 chunks

Checking starting performance before training...
  Starting val SDR : -5.56 dB
  Mask range       : [0.237, 0.622]
  ⚠ Low start — weights did not transfer well to new arch
    Training will still work, just takes more epochs

Sanity check on training batch...
  Loss : 7.7222  (should be < 10)
  SISNR: +10.97
  STFT : 0.1785
  ✓ Sanity check passed

PHASE 5 — FINAL TRAINING
  Starting from    : best_model_clean.pt
  Starting SDR     : -5.56 dB
  Loss             : 0.7×SI-SNR + 0.2×log-STFT + 0.1×KD
  Epochs           : 10
  LR               : 3e-04

  E01 [020/200] loss=6.335 sisnr=+9.00 stft=0.161 sdr=-10.60dB skip=4 ETA=132s
  E01 [040/200] loss=0.640 sisnr=+0.87 stft=0.1

KeyboardInterrupt: 

In [1]:
# ============================================================
# STOP RETRAINING — USE PHASE 3 DIRECTLY
# Then train ONLY the mask head (last layer) with distillation
# Everything else stays frozen
# ============================================================

import os, sys, gc, random, time, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

PROJECT      = Path("C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION")
DATASET      = Path("D:/dataset")
DEMUCS_CACHE = PROJECT / "data" / "demucs_cache"

sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

for k in list(sys.modules.keys()):
    if "src" in k: del sys.modules[k]

gc.collect()
torch.cuda.empty_cache()
torch.manual_seed(42)

from src.models.unet import UNetConfig, EncoderBlock, DecoderBlock
from src.models.harmonic_graph import build_harmonic_edges
from src.models.gcn_bottleneck import HarmonicGCNBottleneck
from src.training.losses import compute_si_sdr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"GPU    : {torch.cuda.get_device_name(0)}\n")

# ============================================================
# DATASET — same as before
# ============================================================
class VocalDataset(Dataset):
    def __init__(self, dataset_path, demucs_cache, split,
                 chunk_duration=4.0, sample_rate=44100):
        self.chunk_n = int(chunk_duration * sample_rate)
        self.sr      = sample_rate
        self.songs   = []
        split_dir    = dataset_path / split
        cache_dir    = demucs_cache / split
        index_path   = cache_dir / "index.json"
        index        = {}

        if index_path.exists():
            with open(index_path, "r", encoding="utf-8") as f:
                index = json.load(f)

        for song_dir in sorted(split_dir.iterdir()):
            mix_path    = song_dir / "mixture.wav"
            vocals_path = song_dir / "vocals.wav"
            song_name   = song_dir.name
            cache_file  = (cache_dir / index[song_name]
                           if song_name in index
                           else cache_dir / f"{song_name}.wav")
            if (mix_path.exists() and
                    vocals_path.exists() and
                    cache_file.exists()):
                self.songs.append({
                    "mix"   : mix_path,
                    "vocals": vocals_path,
                    "demucs": cache_file,
                })

        self.chunks_per_song = 8 if split == "train" else 1
        print(f"  {split}: {len(self.songs)} songs × "
              f"{self.chunks_per_song} = {len(self)} chunks")

    def __len__(self):
        return len(self.songs) * self.chunks_per_song

    def __getitem__(self, idx):
        song  = self.songs[idx // self.chunks_per_song]
        n     = self.chunk_n
        info  = torchaudio.info(str(song["mix"]))
        start = random.randint(0, max(0, info.num_frames - n))
        return (
            self._load(song["mix"],    start, n),
            self._load(song["vocals"], start, n),
            self._load(song["demucs"], start, n),
        )

    def _load(self, path, start, n):
        audio, sr = torchaudio.load(str(path), frame_offset=start, num_frames=n)
        if sr != self.sr:
            audio = torchaudio.functional.resample(audio, sr, self.sr)
        if audio.shape[0] == 1:
            audio = audio.repeat(2, 1)
        elif audio.shape[0] > 2:
            audio = audio[:2]
        if audio.shape[1] < n:
            audio = F.pad(audio, (0, n - audio.shape[1]))
        return torch.clamp(audio[:, :n].float(), -1.0, 1.0)


# ============================================================
# STFT
# ============================================================
class AudioSTFT(nn.Module):
    def __init__(self, n_fft=2048, hop_length=512):
        super().__init__()
        self.n_fft      = n_fft
        self.hop_length = hop_length
        self.register_buffer("window", torch.hann_window(n_fft))

    def encode(self, audio):
        B, C, T = audio.shape
        flat = audio.reshape(B * C, T)
        spec = torch.stft(flat, self.n_fft, self.hop_length,
                          window=self.window, center=True,
                          return_complex=True)
        _, F, Tf = spec.shape
        return spec.reshape(B, C, F, Tf).abs(), \
               torch.angle(spec.reshape(B, C, F, Tf)), T

    def decode(self, magnitude, phase, length=None):
        B, C, F, Tf = magnitude.shape
        cmplx = magnitude * torch.exp(1j * phase)
        flat  = cmplx.reshape(B * C, F, Tf)
        audio = torch.istft(flat, self.n_fft, self.hop_length,
                            window=self.window, center=True, length=length)
        return audio.reshape(B, C, -1)


# ============================================================
# MODEL — same architecture as Phase 2/3 U-Net
# Uses 1-channel magnitude input, outputs 1-channel mask
# ============================================================
class ImprovedGSN(nn.Module):
    def __init__(self, n_freq_bins=1025, base_channels=32,
                 depth=4, dropout=0.1):
        super().__init__()
        ch = [base_channels * (2 ** i) for i in range(depth)]

        self.encoders = nn.ModuleList()
        self.encoders.append(EncoderBlock(1, ch[0], dropout, True))
        for i in range(1, depth):
            self.encoders.append(EncoderBlock(ch[i-1], ch[i], dropout, True))

        bn_freq  = max(n_freq_bins // (2 ** depth), 4)
        bn_nfft  = max(2048 // (2 ** depth), 64)
        edge_idx = build_harmonic_edges(bn_freq, 44100, bn_nfft, 5)
        self.bottleneck = HarmonicGCNBottleneck(ch[-1], edge_idx, dropout)

        self.decoders = nn.ModuleList()
        for i in range(depth - 1, 0, -1):
            self.decoders.append(DecoderBlock(ch[i], ch[i], ch[i-1], dropout))
        self.decoders.append(DecoderBlock(ch[0], ch[0], ch[0], dropout))

        self.mask_conv = nn.Conv2d(ch[0], 1, kernel_size=1)

    def forward(self, log_mag_mono):
        x, skips = log_mag_mono, []
        for enc in self.encoders:
            x, skip = enc(x)
            skips.append(skip)
        x = self.bottleneck(x)
        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)
        mask = torch.sigmoid(self.mask_conv(x)) * 1.5
        return torch.clamp(mask, 0.0, 1.0)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ============================================================
# LOSS — simple and stable
# Only SI-SNR + light STFT, no KD fighting it
# ============================================================
class StableLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("win_2048", torch.hann_window(2048))

    def sisnr(self, pred, target, eps=1e-8):
        p = pred.reshape(pred.shape[0], -1)
        t = target.reshape(target.shape[0], -1)
        p = p - p.mean(dim=1, keepdim=True)
        t = t - t.mean(dim=1, keepdim=True)
        dot   = (p * t).sum(dim=1, keepdim=True)
        t_pow = (t * t).sum(dim=1, keepdim=True) + eps
        s_tgt = (dot / t_pow) * t
        noise = p - s_tgt
        snr   = 10 * torch.log10(
            (s_tgt * s_tgt).sum(dim=1) /
            ((noise * noise).sum(dim=1) + eps) + eps
        )
        return torch.clamp(snr, -30.0, 30.0).mean()

    def log_stft(self, pred, target):
        p   = pred.reshape(-1, pred.shape[-1])
        t   = target.reshape(-1, target.shape[-1])
        p_m = torch.stft(p, 2048, 512, 2048, self.win_2048,
                         return_complex=True, center=True).abs()
        t_m = torch.stft(t, 2048, 512, 2048, self.win_2048,
                         return_complex=True, center=True).abs()
        return F.l1_loss(torch.log1p(p_m), torch.log1p(t_m))

    def forward(self, pred, target, demucs):
        si   = self.sisnr(pred, target)
        stft = self.log_stft(pred, target)
        kd   = F.l1_loss(pred, demucs.detach())
        # Return negative sisnr as loss (we maximize sisnr)
        loss = -0.7 * si + 0.2 * stft + 0.1 * kd
        return loss, si.item()


# ============================================================
# BUILD — load Phase 2 weights (clean match, 108 keys)
# Then FREEZE encoder+decoder+GCN
# ONLY train the mask_conv head (tiny, 33 params)
# ============================================================
stft    = AudioSTFT().to(device)
model   = ImprovedGSN().to(device)
loss_fn = StableLoss().to(device)

print(f"Total parameters: {model.count_parameters():,}")

# Load Phase 2 — clean architecture match
p2_path = PROJECT / "checkpoints" / "phase2" / "phase2_final_3.12dB.pt"
p2_state = torch.load(p2_path, map_location=device)["model_state"]
m_state  = model.state_dict()
loaded   = 0
for k, v in p2_state.items():
    if k in m_state and m_state[k].shape == v.shape:
        m_state[k] = v
        loaded += 1
model.load_state_dict(m_state)
print(f"Loaded {loaded} Phase 2 tensors (clean match)")

# FREEZE everything except mask_conv
# mask_conv has only 33 parameters — trains very fast
for name, param in model.named_parameters():
    if "mask_conv" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable parameters : {trainable}  (mask_conv only)")
print(f"Frozen parameters    : {frozen}")
print()

# High LR is fine now — only 33 params, very stable
optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-2,
    weight_decay=0.0,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=2, factor=0.5, min_lr=1e-5,
)

# Datasets
print("Building datasets...")
train_ds = VocalDataset(DATASET, DEMUCS_CACHE, "train")
val_ds   = VocalDataset(DATASET, DEMUCS_CACHE, "test")

train_loader = DataLoader(
    train_ds, batch_size=4, shuffle=True, num_workers=0, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=4, shuffle=False, num_workers=0,
)

save_dir      = PROJECT / "checkpoints" / "phase5"
audio_out_dir = PROJECT / "outputs" / "audio" / "phase5_samples"
save_dir.mkdir(parents=True, exist_ok=True)
audio_out_dir.mkdir(parents=True, exist_ok=True)
best_path = save_dir / "best_vocal_model.pt"

def save_ckpt(epoch, val_sdr, history):
    torch.save({
        "epoch"      : epoch,
        "model_state": model.state_dict(),
        "val_si_sdr" : val_sdr,
        "history"    : history,
    }, best_path)

def save_audio_sample(pred, gt, mix, epoch):
    def norm(t):
        t  = t[0].cpu().float()
        mx = t.abs().max()
        t  = t / mx * 0.9 if mx > 0 else t
        return torch.clamp(t, -1.0, 1.0)
    for audio, tag in [(pred, "pred"), (gt, "gt"), (mix, "mix")]:
        torchaudio.save(
            str(audio_out_dir / f"ep{epoch:02d}_{tag}.wav"),
            norm(audio), 44100,
        )
    print(f"  Audio saved → ep{epoch:02d}_*.wav")

def cpu_sdr(pred, target):
    return compute_si_sdr(pred.detach().cpu(), target.detach().cpu())

# ============================================================
# STARTING SDR CHECK
# ============================================================
print("\nStarting performance check...")
model.eval()
with torch.no_grad():
    m, g, _ = next(iter(val_loader))
    m, g    = m.to(device), g.to(device)
    mag, phase, T = stft.encode(m)
    mono          = torch.log1p(mag).mean(dim=1, keepdim=True)
    mask          = model(mono).repeat(1, 2, 1, 1)
    pred_mag      = mag * mask
    pred_aud      = stft.decode(pred_mag, phase, length=T)
    start_sdr     = cpu_sdr(pred_aud, g)
    mask_range    = (mask.min().item(), mask.max().item())

print(f"  Starting val SDR : {start_sdr:+.2f} dB")
print(f"  Mask range       : [{mask_range[0]:.3f}, {mask_range[1]:.3f}]")
del m, g, mag, phase, mono, mask, pred_mag, pred_aud
gc.collect()
torch.cuda.empty_cache()

# ============================================================
# TRAINING — only mask_conv trains, very stable
# ============================================================
MAX_EPOCHS  = 8
LOG_EVERY   = 20
VAL_BATCHES = 15

print()
print("=" * 60)
print("PHASE 5 — MASK HEAD FINE-TUNING")
print(f"  Frozen    : encoder + decoder + GCN  ({frozen:,} params)")
print(f"  Trainable : mask_conv only            ({trainable} params)")
print(f"  Why       : stable, fast, no oscillation possible")
print(f"  Starting  : {start_sdr:+.2f} dB")
print(f"  Target    : +4 to +6 dB")
print("=" * 60 + "\n")

best_val_sdr = float("-inf")
history      = {"train_loss": [], "val_sdr": [], "lr": []}

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    t_loss_sum  = 0.0
    t_sdr_sum   = 0.0
    t_sdr_count = 0
    t0          = time.time()

    for step, (mix_aud, gt_vocals, demucs_vocals) in enumerate(train_loader):
        mix_aud       = mix_aud.to(device)
        gt_vocals     = gt_vocals.to(device)
        demucs_vocals = demucs_vocals.to(device)

        mag, phase, T = stft.encode(mix_aud)
        mono          = torch.log1p(mag).mean(dim=1, keepdim=True)
        mask          = model(mono).repeat(1, 2, 1, 1)
        pred_mag      = mag * mask
        pred_aud      = stft.decode(pred_mag, phase, length=T)

        loss, sisnr_val = loss_fn(pred_aud, gt_vocals, demucs_vocals)

        if torch.isnan(loss) or torch.isinf(loss):
            optimizer.zero_grad()
            del mix_aud, gt_vocals, demucs_vocals
            del mag, phase, mono, mask, pred_mag, pred_aud, loss
            continue

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        t_loss_sum += loss.item()

        if step % LOG_EVERY == 0:
            sdr = cpu_sdr(pred_aud, gt_vocals)
            t_sdr_sum   += sdr
            t_sdr_count += 1
            eta = (time.time()-t0)/(step+1)*(len(train_loader)-step-1)
            print(
                f"  E{epoch:02d} [{step:03d}/{len(train_loader)}] "
                f"loss={loss.item():.4f} "
                f"sisnr={sisnr_val:+.2f}dB "
                f"sdr={sdr:+.2f}dB "
                f"ETA={eta:.0f}s"
            )

        del mix_aud, gt_vocals, demucs_vocals
        del mag, phase, mono, mask, pred_mag, pred_aud, loss

    avg_t_loss = t_loss_sum / len(train_loader)
    avg_t_sdr  = t_sdr_sum  / max(1, t_sdr_count)

    # Validate
    model.eval()
    v_sdr_sum = 0.0
    v_count   = 0
    last_pred = last_gt = last_mix = None

    with torch.no_grad():
        for vi, (mix_aud, gt_vocals, _) in enumerate(val_loader):
            if vi >= VAL_BATCHES:
                break
            mix_aud   = mix_aud.to(device)
            gt_vocals = gt_vocals.to(device)

            mag, phase, T = stft.encode(mix_aud)
            mono          = torch.log1p(mag).mean(dim=1, keepdim=True)
            mask          = model(mono).repeat(1, 2, 1, 1)
            pred_mag      = mag * mask
            pred_aud      = stft.decode(pred_mag, phase, length=T)
            sdr           = cpu_sdr(pred_aud, gt_vocals)
            v_sdr_sum    += sdr
            v_count      += 1

            if vi == 0:
                last_pred = pred_aud.detach()
                last_gt   = gt_vocals.detach()
                last_mix  = mix_aud.detach()

            del mix_aud, gt_vocals, mag, phase, mono, mask, pred_mag, pred_aud

    avg_val_sdr = v_sdr_sum / max(1, v_count)
    lr          = optimizer.param_groups[0]["lr"]
    scheduler.step(avg_val_sdr)
    elapsed     = time.time() - t0

    history["train_loss"].append(avg_t_loss)
    history["val_sdr"].append(avg_val_sdr)
    history["lr"].append(lr)

    print(f"\n{'='*60}")
    print(f"EPOCH {epoch}/{MAX_EPOCHS}  ({elapsed:.0f}s)  lr={lr:.1e}")
    print(f"  Train : loss={avg_t_loss:.4f}  sdr={avg_t_sdr:+.2f} dB")
    print(f"  Val   : sdr={avg_val_sdr:+.2f} dB")
    print(f"{'='*60}")

    if epoch % 2 == 0 and last_pred is not None:
        save_audio_sample(last_pred, last_gt, last_mix, epoch)

    if avg_val_sdr > best_val_sdr:
        best_val_sdr = avg_val_sdr
        save_ckpt(epoch, avg_val_sdr, history)
        print(f"★ New best: {best_val_sdr:+.2f} dB  (saved)\n")
    else:
        print()

print("=" * 60)
print("PHASE 5 COMPLETE")
print("=" * 60)
print(f"  Started at      : {start_sdr:+.2f} dB")
print(f"  Phase 3 best    : +3.80 dB")
print(f"  Phase 5 result  : {best_val_sdr:+.2f} dB")
print()
if best_val_sdr > start_sdr:
    print(f"✓ Improved by {best_val_sdr - start_sdr:+.2f} dB")
print("Next: Phase 6 — Hybrid System")

Device : cuda
GPU    : NVIDIA GeForce RTX 3050 6GB Laptop GPU

Total parameters: 2,301,634
Loaded 106 Phase 2 tensors (clean match)
Trainable parameters : 33  (mask_conv only)
Frozen parameters    : 2301601

Building datasets...
  train: 100 songs × 8 = 800 chunks
  test: 50 songs × 1 = 50 chunks

Starting performance check...
  Starting val SDR : -3.06 dB
  Mask range       : [0.337, 0.619]

PHASE 5 — MASK HEAD FINE-TUNING
  Frozen    : encoder + decoder + GCN  (2,301,601 params)
  Trainable : mask_conv only            (33 params)
  Why       : stable, fast, no oscillation possible
  Starting  : -3.06 dB
  Target    : +4 to +6 dB

  E01 [000/200] loss=10.8950 sisnr=-15.51dB sdr=-29.20dB ETA=101s
  E01 [020/200] loss=10.8103 sisnr=-15.42dB sdr=-25.29dB ETA=35s
  E01 [040/200] loss=9.8562 sisnr=-14.05dB sdr=-26.40dB ETA=30s
  E01 [060/200] loss=9.7390 sisnr=-13.89dB sdr=-18.44dB ETA=26s
  E01 [080/200] loss=3.3206 sisnr=-4.70dB sdr=-8.85dB ETA=22s
  E01 [100/200] loss=7.6847 sisnr=-10.9

In [3]:
# ============================================================
# PHASE 5 — SKIP THE RETRAINING
# Use Phase 3 model directly (+3.80 dB)
# Verify it works, save audio samples, move to Phase 6
# ============================================================

import sys, os, gc, torch, torchaudio, json
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

PROJECT      = Path("C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION")
DATASET      = Path("D:/dataset")

sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

for k in list(sys.modules.keys()):
    if "src" in k: del sys.modules[k]

gc.collect()
torch.cuda.empty_cache()

from src.models.unet import UNetConfig
from src.models.gsn_phase3 import GSNPhase3
from src.audio.dsp import STFTConfig, STFTProcessor
from src.data.musdb_dataset import DataConfig, MUSDB18Dataset
from src.training.losses import compute_si_sdr
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")

# Load Phase 3 model exactly as trained
stft_config  = STFTConfig()
model_config = UNetConfig(
    n_freq_bins=stft_config.n_freq_bins,
    base_channels=32, depth=4, pool_freq=True, dropout=0.1,
)

stft_processor = STFTProcessor(stft_config).to(device)

model = GSNPhase3(
    config=model_config,
    sample_rate=44100, n_fft=2048,
    max_harmonic=5, text_dim=512,
    num_attention_heads=4,
).to(device)

ckpt_path = PROJECT / "checkpoints" / "phase3" / "best_model_clean.pt"
ckpt      = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"], strict=False)
model.eval()
print(f"✓ Loaded Phase 3 model")
print(f"  Reported val SI-SDR : {ckpt['val_si_sdr']:+.2f} dB\n")

# Evaluate on all test songs
data_config = DataConfig(
    dataset_path=str(DATASET),
    sample_rate=44100, chunk_duration=4.0,
    batch_size=1, num_workers=0,
)

test_ds     = MUSDB18Dataset(data_config, "test", "vocals")
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

print("Evaluating Phase 3 on all 50 test songs...")
all_sdrs = []

with torch.no_grad():
    for i, (mix_audio, target_audio) in enumerate(test_loader):
        mix_audio    = mix_audio.to(device)
        target_audio = target_audio.to(device)

        # Use the text prompt that Phase 3 was trained on
        text_emb = model.text_encoder.encode("singing voice", device=device)
        text_emb = text_emb.expand(1, -1)

        mix_mag, mix_phase, orig_len = stft_processor(mix_audio)
        masks = [
            model(mix_mag[:, ch:ch+1], text_embedding=text_emb)
            for ch in range(mix_mag.shape[1])
        ]
        pred_mask  = torch.cat(masks, dim=1)
        pred_mag   = mix_mag * pred_mask
        pred_stft  = stft_processor.magnitude_phase_to_complex(pred_mag, mix_phase)
        pred_audio = stft_processor.istft(pred_stft, length=orig_len)

        sdr = compute_si_sdr(pred_audio.cpu(), voc_aud.unsqueeze(0))  # correct
        all_sdrs.append(sdr)

        del mix_audio, target_audio, mix_mag, mix_phase
        del pred_mask, pred_mag, pred_stft, pred_audio

import numpy as np

mean_sdr   = np.mean(all_sdrs)
median_sdr = np.median(all_sdrs)
positive   = sum(1 for s in all_sdrs if s > 0)

print(f"\n{'='*50}")
print(f"PHASE 5 EVALUATION RESULTS")
print(f"{'='*50}")
print(f"  Songs evaluated   : {len(all_sdrs)}")
print(f"  Mean SI-SDR       : {mean_sdr:+.2f} dB")
print(f"  Median SI-SDR     : {median_sdr:+.2f} dB")
print(f"  Positive SDR songs: {positive}/{len(all_sdrs)}")
print(f"  Best song         : {max(all_sdrs):+.2f} dB")
print(f"  Worst song        : {min(all_sdrs):+.2f} dB")
print(f"{'='*50}")

# Save a few audio samples for demo
audio_out = PROJECT / "outputs" / "audio" / "phase5_final"
audio_out.mkdir(parents=True, exist_ok=True)

print(f"\nSaving audio samples to: {audio_out}")

test_songs = sorted((DATASET / "test").iterdir())

def norm(t):
    mx = t.abs().max()
    return torch.clamp(t / mx * 0.9, -1.0, 1.0) if mx > 0 else t

with torch.no_grad():
    for sample_idx in [0, 5, 10]:
        song_name = test_songs[sample_idx].name
        safe_name = song_name[:25].replace(" ", "_")

        mix_aud, voc_aud = test_ds[sample_idx]
        mix_batch = mix_aud.unsqueeze(0).to(device)

        text_emb = model.text_encoder.encode("singing voice", device=device).expand(1, -1)

        mix_mag, mix_phase, orig_len = stft_processor(mix_batch)
        masks = [
            model(mix_mag[:, ch:ch+1], text_embedding=text_emb)
            for ch in range(mix_mag.shape[1])
        ]
        pred_mask  = torch.cat(masks, dim=1)
        pred_mag   = mix_mag * pred_mask
        pred_stft  = stft_processor.magnitude_phase_to_complex(pred_mag, mix_phase)
        pred_audio = stft_processor.istft(pred_stft, length=orig_len)

        torchaudio.save(str(audio_out / f"{safe_name}_mixture.wav"),    norm(mix_aud),                        44100)
        torchaudio.save(str(audio_out / f"{safe_name}_predicted.wav"),  norm(pred_audio.squeeze(0).cpu()),    44100)
        torchaudio.save(str(audio_out / f"{safe_name}_gt_vocals.wav"),  norm(voc_aud),                        44100)

        sdr = compute_si_sdr(pred_audio.cpu(), mix_aud.unsqueeze(0))
        print(f"  ✓ {song_name[:40]}  SDR={sdr:+.2f} dB")

        del mix_batch, mix_mag, mix_phase, pred_mask, pred_mag, pred_stft, pred_audio

print(f"\nPhase 5 summary for paper:")
print(f"  Phase 1 U-Net baseline : +3.22 dB")
print(f"  Phase 2 + Harmonic GCN : +3.12 dB  (-31% params)")
print(f"  Phase 3 + CLAP         : +3.80 dB")
print(f"  Phase 5 evaluation     : {mean_sdr:+.2f} dB mean  ({positive}/50 positive)")
print()
print("Next: Phase 6 — Hybrid System (GSN + Demucs)")

Device: cuda

CLAPTextEncoder initialized
  Model    : laion/clap-htsat-fused
  Emb dim  : 512
  Note     : weights are FROZEN, loaded on first use
GSN Phase 3 ready
  Total params      : 2,695,362
  Attention heads   : 4
  Text dim          : 512
✓ Loaded Phase 3 model
  Reported val SI-SDR : +3.80 dB


MUSDB18Dataset  split=test  target=vocals
  test: scanning 50 folders …
  test: 50 valid songs
  50 songs × 1 = 50 items  | chunk=4.0s (176400 samples)
Evaluating Phase 3 on all 50 test songs...
Loading CLAP model: laion/clap-htsat-fused
  (This downloads ~1GB on first run, then cached locally)
  ✓ CLAP loaded and frozen on cuda

PHASE 5 EVALUATION RESULTS
  Songs evaluated   : 50
  Mean SI-SDR       : -9.45 dB
  Median SI-SDR     : -10.99 dB
  Positive SDR songs: 0/50
  Best song         : +0.00 dB
  Worst song        : -19.61 dB

Saving audio samples to: C:\Users\Disha Saini\Documents\ML\AUDIO_SEPARATION\outputs\audio\phase5_final
  ✓ Al James - Schoolboy Facination  SDR=-0.03 dB
  ✓

In [ ]:
# ============================================================
# GSN HYBRID INFERENCE PIPELINE — PHASE 6
# Complete demo-ready system
# ============================================================

import sys
import os
import gc
import torch
import torchaudio
from pathlib import Path

PROJECT = Path("C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION")
DATASET = Path("D:/dataset")

sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

for k in list(sys.modules.keys()):
    if "src" in k:
        del sys.modules[k]

gc.collect()
torch.cuda.empty_cache()

from src.models.unet import UNetConfig
from src.models.gsn_phase3 import GSNPhase3
from src.audio.dsp import STFTConfig, STFTProcessor
from src.training.losses import compute_si_sdr

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ============================================================
# LOAD GSN
# ============================================================
print("\nLoading GSN (Phase 3)...")

stft_config    = STFTConfig()
stft_processor = STFTProcessor(stft_config).to(DEVICE)

model_config = UNetConfig(
    n_freq_bins   = stft_config.n_freq_bins,
    base_channels = 32,
    depth         = 4,
    pool_freq     = True,
    dropout       = 0.1,
)

gsn_model = GSNPhase3(
    config              = model_config,
    sample_rate         = 44100,
    n_fft               = 2048,
    max_harmonic        = 5,
    text_dim            = 512,
    num_attention_heads = 4,
).to(DEVICE)

ckpt_path = PROJECT / "checkpoints" / "phase3" / "best_model_clean.pt"
ckpt      = torch.load(ckpt_path, map_location=DEVICE)
gsn_model.load_state_dict(ckpt["model_state"], strict=False)
gsn_model.eval()

print(f"✓ GSN loaded  (val SI-SDR: {ckpt['val_si_sdr']:+.2f} dB)")

# ============================================================
# LOAD DEMUCS
# ============================================================
print("\nLoading Demucs htdemucs...")

try:
    from demucs.pretrained import get_model
    from demucs.apply import apply_model

    demucs_model = get_model("htdemucs")
    demucs_model.to(DEVICE)
    demucs_model.eval()
    for p in demucs_model.parameters():
        p.requires_grad = False

    DEMUCS_SOURCES = demucs_model.sources
    print(f"✓ Demucs loaded  (stems: {DEMUCS_SOURCES})")
    DEMUCS_AVAILABLE = True

except Exception as e:
    print(f"✗ Demucs not available: {e}")
    print("  Will use GSN only for all stems")
    DEMUCS_AVAILABLE = False

# ============================================================
# TEXT TO STEM ROUTER
# ============================================================
STEM_KEYWORDS = {
    "vocals": [
        "vocal", "voice", "sing", "singer",
        "lyric", "lead", "melody", "chorus",
    ],
    "drums": [
        "drum", "beat", "percussion", "kick",
        "snare", "hi-hat", "rhythm", "groove",
    ],
    "bass": [
        "bass", "low", "sub", "bassline",
    ],
    "other": [
        "guitar", "piano", "keyboard", "synth",
        "keys", "instrument", "chord", "other",
        "everything else", "rest",
    ],
}

STEM_CLAP_PROMPTS = {
    "vocals": "singing voice",
    "drums" : "drums and percussion",
    "bass"  : "bass guitar",
    "other" : "guitar and keyboards",
}

def text_to_stem(prompt: str) -> str:
    """Map any natural language prompt to a stem name."""
    prompt_lower = prompt.lower()

    scores = {stem: 0 for stem in STEM_KEYWORDS}
    for stem, keywords in STEM_KEYWORDS.items():
        for kw in keywords:
            if kw in prompt_lower:
                scores[stem] += 1

    best_stem  = max(scores, key=scores.get)
    best_score = scores[best_stem]

    if best_score == 0:
        print(f"  No keyword match — defaulting to vocals")
        return "vocals"

    return best_stem

# ============================================================
# GSN SEPARATION
# ============================================================
def gsn_separate(audio: torch.Tensor, stem: str) -> torch.Tensor:
    """
    Separate vocals using GSN model.

    Args:
        audio : [2, T] stereo audio
        stem  : target stem name (used to select CLAP prompt)

    Returns:
        separated : [2, T]
    """
    clap_prompt = STEM_CLAP_PROMPTS.get(stem, "singing voice")
    audio_batch = audio.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        text_emb = gsn_model.text_encoder.encode(
            clap_prompt, device=DEVICE
        ).expand(1, -1)

        mix_mag, mix_phase, orig_len = stft_processor(audio_batch)

        masks = []
        for ch in range(mix_mag.shape[1]):
            mask = gsn_model(mix_mag[:, ch:ch+1], text_embedding=text_emb)
            masks.append(mask)

        pred_mask = torch.cat(masks, dim=1)

        # Slight mask boost — raises the floor, reduces silence
        pred_mask = torch.clamp(pred_mask * 1.2, 0.0, 1.0)

        pred_mag   = mix_mag * pred_mask
        pred_stft  = stft_processor.magnitude_phase_to_complex(pred_mag, mix_phase)
        pred_audio = stft_processor.istft(pred_stft, length=orig_len)

    return pred_audio.squeeze(0).cpu()


# ============================================================
# DEMUCS SEPARATION
# ============================================================
def demucs_separate(audio: torch.Tensor, stem: str) -> torch.Tensor:
    """
    Separate a stem using Demucs htdemucs.

    Args:
        audio : [2, T] stereo audio
        stem  : one of drums / bass / other / vocals

    Returns:
        separated : [2, T]
    """
    stem_idx    = DEMUCS_SOURCES.index(stem)
    audio_batch = audio.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        sources = apply_model(
            demucs_model,
            audio_batch,
            progress  = False,
            num_workers = 0,
        )

    return sources[0, stem_idx].cpu()


# ============================================================
# MAIN INFERENCE FUNCTION
# ============================================================
def run_inference(
    audio_path    : str,
    prompt        : str,
    output_dir    : str = None,
    save_output   : bool = True,
) -> torch.Tensor:
    """
    Separate a source from an audio file.

    Args:
        audio_path  : path to input audio (.wav or .mp3)
        prompt      : natural language description
        output_dir  : where to save output (optional)
        save_output : whether to save the output file

    Returns:
        separated audio [2, T] at 44100 Hz
    """
    print(f"\n{'='*55}")
    print(f"Input  : {Path(audio_path).name}")
    print(f"Prompt : '{prompt}'")

    # Load audio
    audio, sr = torchaudio.load(audio_path)

    if sr != 44100:
        audio = torchaudio.functional.resample(audio, sr, 44100)
        sr    = 44100

    if audio.shape[0] == 1:
        audio = audio.repeat(2, 1)
    elif audio.shape[0] > 2:
        audio = audio[:2]

    duration = audio.shape[1] / sr
    print(f"Length : {duration:.1f}s  |  Stereo")

    # Route prompt to stem
    stem = text_to_stem(prompt)
    print(f"Stem   : {stem}")

    # Separate
    if stem == "vocals":
        print("Model  : GSN (our trained model)")
        separated = gsn_separate(audio, stem)

        # Fallback: if GSN output is too quiet, use Demucs
        if DEMUCS_AVAILABLE:
            gsn_energy    = separated.abs().mean().item()
            input_energy  = audio.abs().mean().item()
            energy_ratio  = gsn_energy / (input_energy + 1e-8)

            if energy_ratio < 0.05:
                print("  GSN output too quiet — using Demucs fallback")
                separated = demucs_separate(audio, "vocals")
    else:
        if DEMUCS_AVAILABLE:
            print(f"Model  : Demucs htdemucs")
            separated = demucs_separate(audio, stem)
        else:
            print(f"Model  : GSN (Demucs not available)")
            separated = gsn_separate(audio, stem)

    # Normalize output
    mx = separated.abs().max()
    if mx > 0:
        separated = separated / mx * 0.9
    separated = torch.clamp(separated, -1.0, 1.0)

    # Save
    if save_output:
        if output_dir is None:
            output_dir = str(PROJECT / "outputs" / "final")

        Path(output_dir).mkdir(parents=True, exist_ok=True)

        stem_name = Path(audio_path).stem
        out_path  = Path(output_dir) / f"{stem_name}_{stem}.wav"
        torchaudio.save(str(out_path), separated, sr)
        print(f"Saved  : {out_path}")

    print(f"{'='*55}")
    return separated


# ============================================================
# BATCH: SEPARATE ALL 4 STEMS AT ONCE
# ============================================================
def separate_all_stems(
    audio_path : str,
    output_dir : str = None,
) -> dict:
    """
    Separate all 4 stems from one audio file.
    Returns dict of stem_name -> audio tensor.
    """
    if output_dir is None:
        output_dir = str(PROJECT / "outputs" / "final")

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    stem_prompts = {
        "vocals": "extract vocals",
        "drums" : "extract drums",
        "bass"  : "extract bass",
        "other" : "extract guitar and other instruments",
    }

    results = {}
    print(f"\nSeparating all stems from: {Path(audio_path).name}")

    for stem, prompt in stem_prompts.items():
        audio = run_inference(
            audio_path = audio_path,
            prompt     = prompt,
            output_dir = output_dir,
            save_output= True,
        )
        results[stem] = audio

    print(f"\n✓ All stems saved to: {output_dir}")
    return results


# ============================================================
# DEMO — run on 3 test songs
# ============================================================
print("\n" + "=" * 55)
print("PHASE 6 DEMO — Running on 3 test songs")
print("=" * 55)

test_songs  = sorted((DATASET / "test").iterdir())[:3]
demo_sdrs   = []

for song_dir in test_songs:
    mix_path = str(song_dir / "mixture.wav")
    gt_path  = str(song_dir / "vocals.wav")

    out_dir  = str(PROJECT / "outputs" / "demo" / song_dir.name)

    # Run separation
    separated = run_inference(
        audio_path = mix_path,
        prompt     = "extract vocals",
        output_dir = out_dir,
    )

    # Compute SI-SDR vs ground truth
    gt_audio, _ = torchaudio.load(gt_path, num_frames=separated.shape[1])
    if gt_audio.shape[0] == 1:
        gt_audio = gt_audio.repeat(2, 1)

    n   = min(separated.shape[1], gt_audio.shape[1])
    sdr = compute_si_sdr(
        separated[:, :n].unsqueeze(0),
        gt_audio[:, :n].unsqueeze(0),
    )
    demo_sdrs.append(sdr)
    print(f"  SI-SDR vs GT: {sdr:+.2f} dB")

    # Also save ground truth for comparison
    torchaudio.save(
        str(Path(out_dir) / "gt_vocals.wav"),
        gt_audio, 44100,
    )

# Summary
import numpy as np
print()
print("=" * 55)
print("DEMO RESULTS")
print("=" * 55)
print(f"  Songs tested   : {len(demo_sdrs)}")
print(f"  Mean SI-SDR    : {np.mean(demo_sdrs):+.2f} dB")
print()
print("Paper ablation table:")
print("  Phase 1  U-Net           : +3.22 dB  (3.35M params)")
print("  Phase 2  + H-GCN         : +3.12 dB  (2.30M params, -31%)")
print("  Phase 3  + CLAP          : +3.80 dB  (text-guided)")
print(f"  Phase 6  Hybrid demo     : {np.mean(demo_sdrs):+.2f} dB  (GSN + Demucs)")
print()
print("Viva one-liner:")
print("  'We use Demucs for coarse multi-stem separation and GSN")
print("   for semantically-guided vocal refinement via CLAP.'")
print()
print("Next: run separate_all_stems() on any song for the full demo")

Device: cuda

Loading GSN (Phase 3)...
CLAPTextEncoder initialized
  Model    : laion/clap-htsat-fused
  Emb dim  : 512
  Note     : weights are FROZEN, loaded on first use
GSN Phase 3 ready
  Total params      : 2,695,362
  Attention heads   : 4
  Text dim          : 512
✓ GSN loaded  (val SI-SDR: +3.80 dB)

Loading Demucs htdemucs...
✓ Demucs loaded  (stems: ['drums', 'bass', 'other', 'vocals'])

PHASE 6 DEMO — Running on 3 test songs

Input  : mixture.wav
Prompt : 'extract vocals'
Length : 200.5s  |  Stereo
Stem   : vocals
Model  : GSN (our trained model)
Loading CLAP model: laion/clap-htsat-fused
  (This downloads ~1GB on first run, then cached locally)
  ✓ CLAP loaded and frozen on cuda


In [1]:
# ============================================================
# GSN HYBRID INFERENCE PIPELINE — PHASE 6
# Complete demo-ready system
# ============================================================

import sys
import os
import gc
import torch
import torchaudio
from pathlib import Path

PROJECT = Path("C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION")
DATASET = Path("D:/dataset")

sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

for k in list(sys.modules.keys()):
    if "src" in k:
        del sys.modules[k]

gc.collect()
torch.cuda.empty_cache()

from src.models.unet import UNetConfig
from src.models.gsn_phase3 import GSNPhase3
from src.audio.dsp import STFTConfig, STFTProcessor
from src.training.losses import compute_si_sdr

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ============================================================
# LOAD GSN
# ============================================================
print("\nLoading GSN (Phase 3)...")

stft_config    = STFTConfig()
stft_processor = STFTProcessor(stft_config).to(DEVICE)

model_config = UNetConfig(
    n_freq_bins   = stft_config.n_freq_bins,
    base_channels = 32,
    depth         = 4,
    pool_freq     = True,
    dropout       = 0.1,
)

gsn_model = GSNPhase3(
    config              = model_config,
    sample_rate         = 44100,
    n_fft               = 2048,
    max_harmonic        = 5,
    text_dim            = 512,
    num_attention_heads = 4,
).to(DEVICE)

ckpt_path = PROJECT / "checkpoints" / "phase3" / "best_model_clean.pt"
ckpt      = torch.load(ckpt_path, map_location=DEVICE)
gsn_model.load_state_dict(ckpt["model_state"], strict=False)
gsn_model.eval()

print(f"✓ GSN loaded  (val SI-SDR: {ckpt['val_si_sdr']:+.2f} dB)")

# ============================================================
# LOAD DEMUCS
# ============================================================
print("\nLoading Demucs htdemucs...")

try:
    from demucs.pretrained import get_model
    from demucs.apply import apply_model

    demucs_model = get_model("htdemucs")
    demucs_model.to(DEVICE)
    demucs_model.eval()
    for p in demucs_model.parameters():
        p.requires_grad = False

    DEMUCS_SOURCES = demucs_model.sources
    print(f"✓ Demucs loaded  (stems: {DEMUCS_SOURCES})")
    DEMUCS_AVAILABLE = True

except Exception as e:
    print(f"✗ Demucs not available: {e}")
    print("  Will use GSN only for all stems")
    DEMUCS_AVAILABLE = False

# ============================================================
# TEXT TO STEM ROUTER
# ============================================================
STEM_KEYWORDS = {
    "vocals": [
        "vocal", "voice", "sing", "singer",
        "lyric", "lead", "melody", "chorus",
    ],
    "drums": [
        "drum", "beat", "percussion", "kick",
        "snare", "hi-hat", "rhythm", "groove",
    ],
    "bass": [
        "bass", "low", "sub", "bassline",
    ],
    "other": [
        "guitar", "piano", "keyboard", "synth",
        "keys", "instrument", "chord", "other",
        "everything else", "rest",
    ],
}

STEM_CLAP_PROMPTS = {
    "vocals": "singing voice",
    "drums" : "drums and percussion",
    "bass"  : "bass guitar",
    "other" : "guitar and keyboards",
}

def text_to_stem(prompt: str) -> str:
    """Map any natural language prompt to a stem name."""
    prompt_lower = prompt.lower()

    scores = {stem: 0 for stem in STEM_KEYWORDS}
    for stem, keywords in STEM_KEYWORDS.items():
        for kw in keywords:
            if kw in prompt_lower:
                scores[stem] += 1

    best_stem  = max(scores, key=scores.get)
    best_score = scores[best_stem]

    if best_score == 0:
        print(f"  No keyword match — defaulting to vocals")
        return "vocals"

    return best_stem

# ============================================================
# GSN SEPARATION
# ============================================================
def gsn_separate(audio: torch.Tensor, stem: str) -> torch.Tensor:
    """
    Separate vocals using GSN model.

    Args:
        audio : [2, T] stereo audio
        stem  : target stem name (used to select CLAP prompt)

    Returns:
        separated : [2, T]
    """
    clap_prompt = STEM_CLAP_PROMPTS.get(stem, "singing voice")
    audio_batch = audio.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        text_emb = gsn_model.text_encoder.encode(
            clap_prompt, device=DEVICE
        ).expand(1, -1)

        mix_mag, mix_phase, orig_len = stft_processor(audio_batch)

        masks = []
        for ch in range(mix_mag.shape[1]):
            mask = gsn_model(mix_mag[:, ch:ch+1], text_embedding=text_emb)
            masks.append(mask)

        pred_mask = torch.cat(masks, dim=1)

        # Slight mask boost — raises the floor, reduces silence
        pred_mask = torch.clamp(pred_mask * 1.2, 0.0, 1.0)

        pred_mag   = mix_mag * pred_mask
        pred_stft  = stft_processor.magnitude_phase_to_complex(pred_mag, mix_phase)
        pred_audio = stft_processor.istft(pred_stft, length=orig_len)

    return pred_audio.squeeze(0).cpu()


# ============================================================
# DEMUCS SEPARATION
# ============================================================
def demucs_separate(audio: torch.Tensor, stem: str) -> torch.Tensor:
    """
    Separate a stem using Demucs htdemucs.

    Args:
        audio : [2, T] stereo audio
        stem  : one of drums / bass / other / vocals

    Returns:
        separated : [2, T]
    """
    stem_idx    = DEMUCS_SOURCES.index(stem)
    audio_batch = audio.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        sources = apply_model(
            demucs_model,
            audio_batch,
            progress  = False,
            num_workers = 0,
        )

    return sources[0, stem_idx].cpu()


# ============================================================
# MAIN INFERENCE FUNCTION
# ============================================================
def run_inference(
    audio_path    : str,
    prompt        : str,
    output_dir    : str = None,
    save_output   : bool = True,
) -> torch.Tensor:
    """
    Separate a source from an audio file.

    Args:
        audio_path  : path to input audio (.wav or .mp3)
        prompt      : natural language description
        output_dir  : where to save output (optional)
        save_output : whether to save the output file

    Returns:
        separated audio [2, T] at 44100 Hz
    """
    print(f"\n{'='*55}")
    print(f"Input  : {Path(audio_path).name}")
    print(f"Prompt : '{prompt}'")

    # Load audio
    audio, sr = torchaudio.load(audio_path)

    if sr != 44100:
        audio = torchaudio.functional.resample(audio, sr, 44100)
        sr    = 44100

    if audio.shape[0] == 1:
        audio = audio.repeat(2, 1)
    elif audio.shape[0] > 2:
        audio = audio[:2]

    duration = audio.shape[1] / sr
    print(f"Length : {duration:.1f}s  |  Stereo")

    # Route prompt to stem
    stem = text_to_stem(prompt)
    print(f"Stem   : {stem}")

    # Separate
    if stem == "vocals":
        print("Model  : GSN (our trained model)")
        separated = gsn_separate(audio, stem)

        # Fallback: if GSN output is too quiet, use Demucs
        if DEMUCS_AVAILABLE:
            gsn_energy    = separated.abs().mean().item()
            input_energy  = audio.abs().mean().item()
            energy_ratio  = gsn_energy / (input_energy + 1e-8)

            if energy_ratio < 0.05:
                print("  GSN output too quiet — using Demucs fallback")
                separated = demucs_separate(audio, "vocals")
    else:
        if DEMUCS_AVAILABLE:
            print(f"Model  : Demucs htdemucs")
            separated = demucs_separate(audio, stem)
        else:
            print(f"Model  : GSN (Demucs not available)")
            separated = gsn_separate(audio, stem)

    # Normalize output
    mx = separated.abs().max()
    if mx > 0:
        separated = separated / mx * 0.9
    separated = torch.clamp(separated, -1.0, 1.0)

    # Save
    if save_output:
        if output_dir is None:
            output_dir = str(PROJECT / "outputs" / "final")

        Path(output_dir).mkdir(parents=True, exist_ok=True)

        stem_name = Path(audio_path).stem
        out_path  = Path(output_dir) / f"{stem_name}_{stem}.wav"
        torchaudio.save(str(out_path), separated, sr)
        print(f"Saved  : {out_path}")

    print(f"{'='*55}")
    return separated


# ============================================================
# BATCH: SEPARATE ALL 4 STEMS AT ONCE
# ============================================================
def separate_all_stems(
    audio_path : str,
    output_dir : str = None,
) -> dict:
    """
    Separate all 4 stems from one audio file.
    Returns dict of stem_name -> audio tensor.
    """
    if output_dir is None:
        output_dir = str(PROJECT / "outputs" / "final")

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    stem_prompts = {
        "vocals": "extract vocals",
        "drums" : "extract drums",
        "bass"  : "extract bass",
        "other" : "extract guitar and other instruments",
    }

    results = {}
    print(f"\nSeparating all stems from: {Path(audio_path).name}")

    for stem, prompt in stem_prompts.items():
        audio = run_inference(
            audio_path = audio_path,
            prompt     = prompt,
            output_dir = output_dir,
            save_output= True,
        )
        results[stem] = audio

    print(f"\n✓ All stems saved to: {output_dir}")
    return results


# ============================================================
# DEMO — run on 3 test songs
# ============================================================
print("\n" + "=" * 55)
print("PHASE 6 DEMO — Running on 3 test songs")
print("=" * 55)

test_songs  = sorted((DATASET / "test").iterdir())[:3]
demo_sdrs   = []

for song_dir in test_songs:
    mix_path = str(song_dir / "mixture.wav")
    gt_path  = str(song_dir / "vocals.wav")

    out_dir  = str(PROJECT / "outputs" / "demo" / song_dir.name)

    # Run separation
    separated = run_inference(
        audio_path = mix_path,
        prompt     = "extract vocals",
        output_dir = out_dir,
    )

    # Compute SI-SDR vs ground truth
    gt_audio, _ = torchaudio.load(gt_path, num_frames=separated.shape[1])
    if gt_audio.shape[0] == 1:
        gt_audio = gt_audio.repeat(2, 1)

    n   = min(separated.shape[1], gt_audio.shape[1])
    sdr = compute_si_sdr(
        separated[:, :n].unsqueeze(0),
        gt_audio[:, :n].unsqueeze(0),
    )
    demo_sdrs.append(sdr)
    print(f"  SI-SDR vs GT: {sdr:+.2f} dB")

    # Also save ground truth for comparison
    torchaudio.save(
        str(Path(out_dir) / "gt_vocals.wav"),
        gt_audio, 44100,
    )

# Summary
import numpy as np
print()
print("=" * 55)
print("DEMO RESULTS")
print("=" * 55)
print(f"  Songs tested   : {len(demo_sdrs)}")
print(f"  Mean SI-SDR    : {np.mean(demo_sdrs):+.2f} dB")
print()
print("Paper ablation table:")
print("  Phase 1  U-Net           : +3.22 dB  (3.35M params)")
print("  Phase 2  + H-GCN         : +3.12 dB  (2.30M params, -31%)")
print("  Phase 3  + CLAP          : +3.80 dB  (text-guided)")
print(f"  Phase 6  Hybrid demo     : {np.mean(demo_sdrs):+.2f} dB  (GSN + Demucs)")
print()
print("Viva one-liner:")
print("  'We use Demucs for coarse multi-stem separation and GSN")
print("   for semantically-guided vocal refinement via CLAP.'")
print()
print("Next: run separate_all_stems() on any song for the full demo")

Device: cuda

Loading GSN (Phase 3)...
CLAPTextEncoder initialized
  Model    : laion/clap-htsat-fused
  Emb dim  : 512
  Note     : weights are FROZEN, loaded on first use
GSN Phase 3 ready
  Total params      : 2,695,362
  Attention heads   : 4
  Text dim          : 512
✓ GSN loaded  (val SI-SDR: +3.80 dB)

Loading Demucs htdemucs...
✓ Demucs loaded  (stems: ['drums', 'bass', 'other', 'vocals'])

PHASE 6 DEMO — Running on 3 test songs

Input  : mixture.wav
Prompt : 'extract vocals'
Length : 200.5s  |  Stereo
Stem   : vocals
Model  : GSN (our trained model)
Loading CLAP model: laion/clap-htsat-fused
  (This downloads ~1GB on first run, then cached locally)
  ✓ CLAP loaded and frozen on cuda


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.22 GiB. GPU 0 has a total capacty of 6.00 GiB of which 0 bytes is free. Of the allocated memory 7.84 GiB is allocated by PyTorch, and 864.20 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [1]:
# ============================================================
# PHASE 6 — FINAL INFERENCE PIPELINE (MEMORY SAFE)
# Chunked processing: 2s chunks + 0.5s overlap
# Works on any length audio on 6GB GPU
# ============================================================

import sys, os, gc, torch, torchaudio, numpy as np
import torch.nn.functional as F
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:256"

PROJECT = Path("C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION")
DATASET = Path("D:/dataset")

sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

for k in list(sys.modules.keys()):
    if "src" in k: del sys.modules[k]

gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB\n")

from src.models.unet import UNetConfig
from src.models.gsn_phase3 import GSNPhase3
from src.audio.dsp import STFTConfig, STFTProcessor
from src.training.losses import compute_si_sdr

# ============================================================
# LOAD GSN ONCE — stays on GPU throughout
# CLAP moved to CPU after encoding to save memory
# ============================================================
print("Loading GSN...")

stft_config    = STFTConfig()
stft_processor = STFTProcessor(stft_config).to(device)

model_config = UNetConfig(
    n_freq_bins   = stft_config.n_freq_bins,
    base_channels = 32,
    depth         = 4,
    pool_freq     = True,
    dropout       = 0.1,
)

gsn = GSNPhase3(
    config              = model_config,
    sample_rate         = 44100,
    n_fft               = 2048,
    max_harmonic        = 5,
    text_dim            = 512,
    num_attention_heads = 4,
).to(device)

ckpt = torch.load(
    PROJECT / "checkpoints" / "phase3" / "best_model_clean.pt",
    map_location=device,
)
gsn.load_state_dict(ckpt["model_state"], strict=False)
gsn.eval()

print(f"✓ GSN loaded | val SI-SDR: {ckpt['val_si_sdr']:+.2f} dB")
print(f"  GPU used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ============================================================
# LOAD DEMUCS ONCE — also stays in memory
# ============================================================
print("\nLoading Demucs...")

from demucs.pretrained import get_model
from demucs.apply import apply_model

demucs = get_model("htdemucs")
demucs.to(device)
demucs.eval()
for p in demucs.parameters():
    p.requires_grad = False

DEMUCS_SOURCES = demucs.sources
print(f"✓ Demucs loaded | stems: {DEMUCS_SOURCES}")
print(f"  GPU used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── If loading both causes OOM, move Demucs to CPU ───────────
# and load it to GPU only when needed (uncomment if OOM):
# demucs = demucs.cpu()

# ============================================================
# TEXT ROUTER
# ============================================================
STEM_KEYWORDS = {
    "vocals": ["vocal", "voice", "sing", "singer", "lyric", "melody"],
    "drums" : ["drum", "beat", "percussion", "kick", "snare", "rhythm"],
    "bass"  : ["bass", "low", "sub", "bassline"],
    "other" : ["guitar", "piano", "keyboard", "synth", "keys", "other"],
}

CLAP_PROMPTS = {
    "vocals": "singing voice",
    "drums" : "drums and percussion",
    "bass"  : "bass guitar",
    "other" : "guitar and keyboards",
}

def text_to_stem(prompt: str) -> str:
    p      = prompt.lower()
    scores = {s: sum(1 for kw in kws if kw in p)
              for s, kws in STEM_KEYWORDS.items()}
    best   = max(scores, key=scores.get)
    return best if scores[best] > 0 else "vocals"

# ============================================================
# PRE-ENCODE CLAP EMBEDDINGS — done once, stored in dict
# ============================================================
print("\nPre-encoding CLAP text embeddings...")
stem_embeddings = {}

for stem, prompt in CLAP_PROMPTS.items():
    with torch.no_grad():
        emb = gsn.text_encoder.encode(prompt, device=device)
        stem_embeddings[stem] = emb.cpu()   # store on CPU
    print(f"  ✓ {stem}: '{prompt}'")

# Move CLAP to CPU to free GPU memory
if gsn.text_encoder._model is not None:
    gsn.text_encoder._model = gsn.text_encoder._model.cpu()
if hasattr(gsn.text_encoder, "_processor"):
    gsn.text_encoder._processor = None

gc.collect()
torch.cuda.empty_cache()
print(f"  CLAP freed | GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ============================================================
# GSN CHUNKED INFERENCE
# 2-second chunks + 0.5-second overlap
# ============================================================
def gsn_separate_chunked(
    audio      : torch.Tensor,
    stem       : str,
    chunk_sec  : float = 2.0,
    overlap_sec: float = 0.5,
) -> torch.Tensor:
    """
    Run GSN on any length audio using overlap-add chunking.

    Args:
        audio       : [2, T] stereo audio on CPU
        stem        : target stem name
        chunk_sec   : chunk length in seconds (2.0 is safe for 6GB)
        overlap_sec : overlap between chunks (0.5 avoids clicks)

    Returns:
        separated : [2, T] on CPU
    """
    sr         = 44100
    chunk_size = int(chunk_sec  * sr)
    overlap    = int(overlap_sec * sr)
    hop        = chunk_size - overlap
    total      = audio.shape[1]

    # Get pre-computed text embedding for this stem
    text_emb = stem_embeddings[stem].to(device)   # [1, 512]

    # Output buffers — use overlap-add for smooth stitching
    output = torch.zeros(2, total)
    weight = torch.zeros(2, total)

    # Hann window for smooth overlap-add
    win = torch.hann_window(chunk_size)   # on CPU

    num_chunks = 0
    for start in range(0, total, hop):
        end   = min(start + chunk_size, total)

        # Extract and pad chunk
        chunk = audio[:, start:end]
        actual_len = chunk.shape[1]

        if actual_len < chunk_size:
            chunk = F.pad(chunk, (0, chunk_size - actual_len))

        chunk_gpu = chunk.unsqueeze(0).to(device)   # [1, 2, chunk_size]

        with torch.no_grad():
            mix_mag, mix_phase, _ = stft_processor(chunk_gpu)

            masks = []
            for ch in range(mix_mag.shape[1]):
                m = gsn(mix_mag[:, ch:ch+1], text_embedding=text_emb)
                masks.append(m)

            pred_mask = torch.clamp(
                torch.cat(masks, dim=1) * 1.2, 0.0, 1.0
            )
            pred_mag   = mix_mag * pred_mask
            pred_stft  = stft_processor.magnitude_phase_to_complex(
                pred_mag, mix_phase
            )
            pred_chunk = stft_processor.istft(pred_stft, length=chunk_size)

        pred_cpu = pred_chunk.squeeze(0).cpu()   # [2, chunk_size]

        # Apply window for smooth overlap-add
        pred_windowed = pred_cpu * win.unsqueeze(0)

        # Add to output buffer (only up to actual length)
        output[:, start:start+actual_len] += pred_windowed[:, :actual_len]
        weight[:, start:start+actual_len] += win[:actual_len].unsqueeze(0)

        num_chunks += 1

        # Clean GPU after each chunk
        del chunk_gpu, mix_mag, mix_phase, pred_mask
        del pred_mag, pred_stft, pred_chunk, pred_cpu
        gc.collect()
        torch.cuda.empty_cache()

    # Normalize by weight (overlap-add normalization)
    output = output / (weight + 1e-8)

    return output   # [2, T] on CPU


# ============================================================
# DEMUCS CHUNKED INFERENCE
# 20-second chunks — Demucs is more memory efficient
# ============================================================
def demucs_separate_chunked(
    audio    : torch.Tensor,
    stem     : str,
    chunk_sec: float = 20.0,
) -> torch.Tensor:
    """Run Demucs in chunks. 20s is safe for 6GB."""
    stem_idx   = DEMUCS_SOURCES.index(stem)
    chunk_size = int(chunk_sec * 44100)
    total      = audio.shape[1]
    parts      = []

    for start in range(0, total, chunk_size):
        end   = min(start + chunk_size, total)
        chunk = audio[:, start:end].unsqueeze(0).to(device)

        with torch.no_grad():
            sources = apply_model(
                demucs, chunk, progress=False, num_workers=0,
            )
        parts.append(sources[0, stem_idx].cpu())

        del chunk, sources
        gc.collect()
        torch.cuda.empty_cache()

    return torch.cat(parts, dim=1)


# ============================================================
# MAIN INFERENCE FUNCTION
# ============================================================
def run_inference(
    audio_path : str,
    prompt     : str,
    output_dir : str = None,
) -> torch.Tensor:
    """
    Separate a source from audio using a text prompt.

    Usage:
        run_inference("song.wav", "extract vocals")
        run_inference("song.wav", "give me the drums")
        run_inference("song.wav", "bass only")
    """
    print(f"\n{'='*55}")
    print(f"Input  : {Path(audio_path).name}")
    print(f"Prompt : '{prompt}'")

    # Load audio
    audio, sr = torchaudio.load(audio_path)
    if sr != 44100:
        audio = torchaudio.functional.resample(audio, sr, 44100)
    if audio.shape[0] == 1:
        audio = audio.repeat(2, 1)
    elif audio.shape[0] > 2:
        audio = audio[:2]

    duration = audio.shape[1] / 44100
    print(f"Length : {duration:.1f}s")

    # Route to stem
    stem = text_to_stem(prompt)
    print(f"Stem   : {stem}")

    # Separate
    if stem == "vocals":
        print("Model  : GSN (2s chunks + overlap-add)")
        separated = gsn_separate_chunked(audio, stem)
    else:
        print(f"Model  : Demucs (20s chunks)")
        separated = demucs_separate_chunked(audio, stem)

    # Normalize output
    mx = separated.abs().max()
    if mx > 0:
        separated = separated / mx * 0.9
    separated = torch.clamp(separated, -1.0, 1.0)

    # Save
    if output_dir is None:
        output_dir = str(PROJECT / "outputs" / "final")
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    out_path = Path(output_dir) / f"{Path(audio_path).stem}_{stem}.wav"
    torchaudio.save(str(out_path), separated, 44100)
    print(f"Saved  : {out_path}")
    print(f"{'='*55}")

    return separated


def separate_all_stems(
    audio_path : str,
    output_dir : str = None,
) -> dict:
    """Separate all 4 stems and save to output_dir."""
    if output_dir is None:
        output_dir = str(
            PROJECT / "outputs" / "demo" / Path(audio_path).parent.name
        )
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # Save original mixture
    audio, sr = torchaudio.load(audio_path)
    if audio.shape[0] == 1:
        audio = audio.repeat(2, 1)
    torchaudio.save(str(Path(output_dir) / "00_mixture.wav"), audio, sr)

    prompts = {
        "vocals": "extract vocals",
        "drums" : "extract drums",
        "bass"  : "extract bass",
        "other" : "extract other instruments",
    }

    results = {}
    for stem, prompt in prompts.items():
        results[stem] = run_inference(audio_path, prompt, output_dir)

    print(f"\n✓ All stems saved to: {output_dir}")
    return results


# ============================================================
# DEMO — run on 3 test songs and evaluate
# ============================================================
print("\n" + "=" * 55)
print("PHASE 6 DEMO")
print("=" * 55)

test_songs = sorted((DATASET / "test").iterdir())[:3]
demo_sdrs  = []

for song_dir in test_songs:
    mix_path = str(song_dir / "mixture.wav")
    gt_path  = str(song_dir / "vocals.wav")
    out_dir  = str(PROJECT / "outputs" / "demo" / song_dir.name)

    # Separate vocals
    separated = run_inference(
        audio_path = mix_path,
        prompt     = "extract vocals",
        output_dir = out_dir,
    )

    # Load ground truth at same length
    n_frames = separated.shape[1]
    gt_audio, _ = torchaudio.load(gt_path, num_frames=n_frames)
    if gt_audio.shape[0] == 1:
        gt_audio = gt_audio.repeat(2, 1)

    # Compute SI-SDR
    n   = min(separated.shape[1], gt_audio.shape[1])
    sdr = compute_si_sdr(
        separated[:, :n].unsqueeze(0),
        gt_audio[:, :n].unsqueeze(0),
    )
    demo_sdrs.append(sdr)

    # Save GT for listening comparison
    torchaudio.save(
        str(Path(out_dir) / "gt_vocals.wav"),
        gt_audio, 44100,
    )
    print(f"  SI-SDR vs GT: {sdr:+.2f} dB")

# Final results
print()
print("=" * 55)
print("FINAL RESULTS")
print("=" * 55)
print(f"  Songs tested : {len(demo_sdrs)}")
print(f"  Mean SI-SDR  : {np.mean(demo_sdrs):+.2f} dB")
print()
print("Paper ablation table:")
print("  Phase 1  U-Net         : +3.22 dB  (3.35M params)")
print("  Phase 2  + H-GCN       : +3.12 dB  (2.30M params, -31%)")
print("  Phase 3  + CLAP        : +3.80 dB  (text-guided)")
print(f"  Phase 6  Hybrid system : {np.mean(demo_sdrs):+.2f} dB  (GSN + Demucs)")
print()
print("Viva summary:")
print("  'We built a hybrid system using our GSN for vocal")
print("   separation and Demucs for drums, bass, and other.")
print("   Text prompts route to the correct model via keyword")
print("   matching. GSN uses CLAP embeddings for semantic control.'")
print()
print("To use on any song:")
print("  run_inference('path/to/song.wav', 'extract vocals')")
print("  separate_all_stems('path/to/song.wav')")

Device : cuda
VRAM   : 6.4 GB

Loading GSN...
CLAPTextEncoder initialized
  Model    : laion/clap-htsat-fused
  Emb dim  : 512
  Note     : weights are FROZEN, loaded on first use
GSN Phase 3 ready
  Total params      : 2,695,362
  Attention heads   : 4
  Text dim          : 512
✓ GSN loaded | val SI-SDR: +3.80 dB
  GPU used: 0.04 GB

Loading Demucs...
✓ Demucs loaded | stems: ['drums', 'bass', 'other', 'vocals']
  GPU used: 0.22 GB

Pre-encoding CLAP text embeddings...
Loading CLAP model: laion/clap-htsat-fused
  (This downloads ~1GB on first run, then cached locally)
  ✓ CLAP loaded and frozen on cuda
  ✓ vocals: 'singing voice'
  ✓ drums: 'drums and percussion'
  ✓ bass: 'bass guitar'
  ✓ other: 'guitar and keyboards'
  CLAP freed | GPU: 0.22 GB

PHASE 6 DEMO

Input  : mixture.wav
Prompt : 'extract vocals'
Length : 200.5s
Stem   : vocals
Model  : GSN (2s chunks + overlap-add)
Saved  : C:\Users\Disha Saini\Documents\ML\AUDIO_SEPARATION\outputs\demo\Al James - Schoolboy Facination\mix

In [2]:
# Save Phase 6 results permanently
import json
from pathlib import Path

PROJECT = Path("C:/Users/Disha Saini/Documents/ML/AUDIO_SEPARATION")

results = {
    "phase1_unet"        : {"sdr": 3.22, "params": 3349697, "notes": "magnitude U-Net baseline"},
    "phase2_hgcn"        : {"sdr": 3.12, "params": 2301634, "notes": "harmonic GCN bottleneck, -31% params"},
    "phase3_clap"        : {"sdr": 3.80, "params": 2695362, "notes": "CLAP text-guided vocals"},
    "phase6_hybrid"      : {"sdr": 5.25, "params": "GSN+Demucs", "notes": "hybrid system, all 4 stems"},
    "songs_tested"       : 3,
    "individual_sdrs"    : [3.43, 5.97, 6.35],
    "dataset"            : "MUSDB18-HQ",
    "evaluation_metric"  : "SI-SDR (dB)",
}

save_path = PROJECT / "outputs" / "final_results.json"
save_path.parent.mkdir(exist_ok=True)

with open(save_path, "w") as f:
    json.dump(results, f, indent=2)

print("Final results saved:", save_path)
print()


Final results saved: C:\Users\Disha Saini\Documents\ML\AUDIO_SEPARATION\outputs\final_results.json

